In [1]:
# Cell 01 — Initialize Architecture-C notebook with early-stopping training policy
# DECISION: ADD

from pathlib import Path
import os
import sys
import json
import math
import time
import random
import hashlib

import numpy as np
import torch

# -------------------------------------------------------------------------
# Run mode
# -------------------------------------------------------------------------
RUN_MODE = "production"

assert RUN_MODE in {
    "fast",
    "production",
}

# -------------------------------------------------------------------------
# Reproducibility
# -------------------------------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -------------------------------------------------------------------------
# Device
# -------------------------------------------------------------------------
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif (
    getattr(torch.backends, "mps", None) is not None
    and torch.backends.mps.is_available()
):
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

# -------------------------------------------------------------------------
# Architecture identity
#
# Architecture C is deliberately recurrent rather than Transformer-based.
# It models the sequence using stacked LSTM recurrent state.
# -------------------------------------------------------------------------
ARCHITECTURE_NAME = (
    "Architecture_C_Deep_LSTM_Language_Model"
)

ARCHITECTURE_VERSION = "C_v1"

EXPERIMENT_ID = (
    f"CS781_Phase1_{ARCHITECTURE_VERSION}_{RUN_MODE}"
)

# -------------------------------------------------------------------------
# Working directories
# -------------------------------------------------------------------------
NOTEBOOK_DIR = Path(
    "/kaggle/working/architecture_C"
)

CHECKPOINTS_DIR = (
    NOTEBOOK_DIR / "checkpoints"
)

CONFIGS_DIR = (
    NOTEBOOK_DIR / "configs"
)

METRICS_DIR = (
    NOTEBOOK_DIR / "metrics"
)

PREDICTIONS_DIR = (
    NOTEBOOK_DIR / "predictions"
)

SUBMISSION_DIR = (
    NOTEBOOK_DIR / "submission"
)

TEST_DATA_DIR = (
    NOTEBOOK_DIR / "test_data"
)

for _directory in (
    NOTEBOOK_DIR,
    CHECKPOINTS_DIR,
    CONFIGS_DIR,
    METRICS_DIR,
    PREDICTIONS_DIR,
    SUBMISSION_DIR,
    TEST_DATA_DIR,
):
    _directory.mkdir(
        parents=True,
        exist_ok=True,
    )

# -------------------------------------------------------------------------
# Architecture configuration
#
# Production:
#   50k vocabulary
#   1280-dimensional token embeddings / recurrent state
#   3 stacked LSTM layers
#
# With tied input/output embeddings this is approximately 103M parameters,
# comfortably inside the required 50M–300M range and near the recommended
# 100M–150M region.
# -------------------------------------------------------------------------
if RUN_MODE == "fast":
    MODEL_CONFIG = {
        "vocab_size": 50_000,
        "context_length": 128,
        "embedding_dim": 256,
        "hidden_size": 384,
        "num_layers": 2,
        "dropout": 0.10,
        "tie_embeddings": True,
        "use_bias": True,
    }

else:
    MODEL_CONFIG = {
        "vocab_size": 50_000,
        "context_length": 1024,
        "embedding_dim": 1280,
        "hidden_size": 1280,
        "num_layers": 3,
        "dropout": 0.10,
        "tie_embeddings": True,
        "use_bias": True,
    }

# -------------------------------------------------------------------------
# Training configuration
#
# The production token count is intentionally a MAXIMUM budget.
# Early stopping can terminate substantially earlier.
# -------------------------------------------------------------------------
if RUN_MODE == "fast":
    TRAINING_CONFIG = {
        "train_tokens": 10_240,
        "validation_tokens": 8_192,
        "micro_batch_size": 2,
        "gradient_accumulation_steps": 2,
        "max_steps": 20,
        "warmup_steps": 2,
        "learning_rate": 3e-4,
        "min_learning_rate": 3e-5,
        "weight_decay": 0.10,
        "grad_clip": 1.0,
        "eval_every_steps": 10,
        "eval_batches": 4,
        "checkpoint_every_steps": 10,
        "log_every_steps": 1,
    }

else:
    TRAINING_CONFIG = {
        # Maximum training budget, not a mandatory stopping point.
        "train_tokens": 100_000_000,
        "validation_tokens": 5_000_000,
        "micro_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "max_steps": 0,  # Derived after context length is known.
        "warmup_steps": 500,
        "learning_rate": 3e-4,
        "min_learning_rate": 3e-5,
        "weight_decay": 0.10,
        "grad_clip": 1.0,
        "eval_every_steps": 500,
        "eval_batches": 100,
        "checkpoint_every_steps": 500,
        "log_every_steps": 10,
    }

# -------------------------------------------------------------------------
# Derive maximum optimizer steps from token budget.
#
# Effective tokens per optimizer step =
# micro_batch × gradient_accumulation × context_length
# -------------------------------------------------------------------------
_EFFECTIVE_TOKENS_PER_STEP = (
    TRAINING_CONFIG["micro_batch_size"]
    * TRAINING_CONFIG["gradient_accumulation_steps"]
    * MODEL_CONFIG["context_length"]
)

if TRAINING_CONFIG["max_steps"] == 0:
    TRAINING_CONFIG["max_steps"] = max(
        1,
        math.ceil(
            TRAINING_CONFIG["train_tokens"]
            / _EFFECTIVE_TOKENS_PER_STEP
        ),
    )

# -------------------------------------------------------------------------
# Early-stopping policy
# -------------------------------------------------------------------------
EARLY_STOPPING_CONFIG = {
    "enabled": True,
    "patience_evaluations": (
        2
        if RUN_MODE == "fast"
        else 4
    ),
    "min_delta": 0.01,
    "minimum_steps_before_stop": (
        10
        if RUN_MODE == "fast"
        else 1_000
    ),
}

# -------------------------------------------------------------------------
# Data mixing
# -------------------------------------------------------------------------
SOURCE_NAMES = (
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
)

SOURCE_MIX = {
    "acts": 0.25,
    "high_courts": 0.25,
    "supreme_court": 0.25,
    "tribunals": 0.25,
}

assert abs(
    sum(SOURCE_MIX.values()) - 1.0
) < 1e-8

# -------------------------------------------------------------------------
# Save configuration immediately.
# -------------------------------------------------------------------------
CONFIG_PATH = (
    CONFIGS_DIR / "architecture_C_config.json"
)

CONFIG_PAYLOAD = {
    "experiment_id": EXPERIMENT_ID,
    "architecture_name": ARCHITECTURE_NAME,
    "architecture_version": ARCHITECTURE_VERSION,
    "run_mode": RUN_MODE,
    "seed": SEED,
    "device": str(DEVICE),
    "model_config": MODEL_CONFIG,
    "training_config": TRAINING_CONFIG,
    "early_stopping": EARLY_STOPPING_CONFIG,
    "source_mix": SOURCE_MIX,
}

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        CONFIG_PAYLOAD,
        f,
        ensure_ascii=False,
        indent=2,
    )

# -------------------------------------------------------------------------
# Runtime summary
# -------------------------------------------------------------------------
print("✅ Architecture-C notebook initialized.")
print(
    f"Architecture       : {ARCHITECTURE_NAME}"
)
print(
    f"Run mode           : {RUN_MODE}"
)
print(
    f"Device             : {DEVICE}"
)
print(
    f"Context length     : "
    f"{MODEL_CONFIG['context_length']}"
)
print(
    f"Max train tokens   : "
    f"{TRAINING_CONFIG['train_tokens']:,}"
)
print(
    f"Max optimizer steps: "
    f"{TRAINING_CONFIG['max_steps']:,}"
)
print(
    f"Early-stop patience: "
    f"{EARLY_STOPPING_CONFIG['patience_evaluations']} evals"
)
print(
    f"Minimum delta      : "
    f"{EARLY_STOPPING_CONFIG['min_delta']}"
)

✅ Architecture-C notebook initialized.
Architecture       : Architecture_C_Deep_LSTM_Language_Model
Run mode           : production
Device             : cuda
Context length     : 1024
Max train tokens   : 100,000,000
Max optimizer steps: 12,208
Early-stop patience: 4 evals
Minimum delta      : 0.01


In [2]:
# Cell 02 — Locate and register the persistent Phase-1 tokenized corpus
# DECISION: ADD

from pathlib import Path

# -------------------------------------------------------------------------
# Search Kaggle input mounts for the persistent tokenized corpus created by
# Notebook 1. We do not redownload or retokenize the official corpus.
# -------------------------------------------------------------------------
KAGGLE_INPUT_ROOT = Path(
    "/kaggle/input"
)

DATASET_ROOT = None

for metadata_path in KAGGLE_INPUT_ROOT.rglob(
    "dataset_metadata.json"
):
    candidate_root = (
        metadata_path.parent.parent
    )

    tokenizer_candidate = (
        candidate_root
        / "tokenizer"
        / "tokenizer.json"
    )

    tokenized_candidate = (
        candidate_root
        / "tokenized"
    )

    if (
        tokenizer_candidate.is_file()
        and tokenized_candidate.is_dir()
    ):
        DATASET_ROOT = candidate_root
        break

assert DATASET_ROOT is not None, (
    "The persistent CS781 Phase-1 tokenized corpus was not found "
    "under /kaggle/input."
)

# -------------------------------------------------------------------------
# Register all required metadata paths.
# -------------------------------------------------------------------------
TOKENIZER_PATH = (
    DATASET_ROOT
    / "tokenizer"
    / "tokenizer.json"
)

DATASET_METADATA_PATH = (
    DATASET_ROOT
    / "metadata"
    / "dataset_metadata.json"
)

TOKENIZER_METADATA_PATH = (
    DATASET_ROOT
    / "metadata"
    / "tokenizer_metadata.json"
)

TOKENIZATION_MANIFEST_PATH = (
    DATASET_ROOT
    / "metadata"
    / "tokenization_manifest.json"
)

TOKENIZED_DIR = (
    DATASET_ROOT
    / "tokenized"
)

# -------------------------------------------------------------------------
# Required-file checks.
# -------------------------------------------------------------------------
for _path in (
    TOKENIZER_PATH,
    DATASET_METADATA_PATH,
    TOKENIZER_METADATA_PATH,
    TOKENIZATION_MANIFEST_PATH,
):
    assert _path.is_file(), (
        f"Required dataset artifact not found: {_path}"
    )

for _source in SOURCE_NAMES:
    for _split in (
        "train",
        "val",
    ):
        _shard = (
            TOKENIZED_DIR
            / f"{_source}_{_split}.bin"
        )

        assert _shard.is_file(), (
            f"Missing tokenized shard: {_shard}"
        )

# -------------------------------------------------------------------------
# Save dataset registration metadata for reproducibility.
# -------------------------------------------------------------------------
DATASET_REGISTRATION = {
    "dataset_root": str(DATASET_ROOT),
    "tokenizer_path": str(TOKENIZER_PATH),
    "dataset_metadata_path": str(
        DATASET_METADATA_PATH
    ),
    "tokenizer_metadata_path": str(
        TOKENIZER_METADATA_PATH
    ),
    "tokenization_manifest_path": str(
        TOKENIZATION_MANIFEST_PATH
    ),
    "tokenized_dir": str(TOKENIZED_DIR),
    "sources": list(SOURCE_NAMES),
}

DATASET_REGISTRATION_PATH = (
    CONFIGS_DIR
    / "dataset_registration.json"
)

with open(
    DATASET_REGISTRATION_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        DATASET_REGISTRATION,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("✅ Phase-1 tokenized corpus registered.")
print(
    f"Dataset root : {DATASET_ROOT}"
)
print(
    f"Tokenizer    : {TOKENIZER_PATH}"
)
print(
    f"Tokenized dir: {TOKENIZED_DIR}"
)
print(
    f"Sources      : {', '.join(SOURCE_NAMES)}"
)

✅ Phase-1 tokenized corpus registered.
Dataset root : /kaggle/input/datasets/rohitbatra2004/cs781-phase1-tokenized-corpus
Tokenizer    : /kaggle/input/datasets/rohitbatra2004/cs781-phase1-tokenized-corpus/tokenizer/tokenizer.json
Tokenized dir: /kaggle/input/datasets/rohitbatra2004/cs781-phase1-tokenized-corpus/tokenized
Sources      : acts, high_courts, supreme_court, tribunals


In [3]:
# Cell 03 — Load the Phase-1 tokenizer and resolve actual special tokens
# DECISION: UPDATE

import json
from tokenizers import Tokenizer

# -------------------------------------------------------------------------
# Load the exact tokenizer trained for Phase 1.
# -------------------------------------------------------------------------
tokenizer = Tokenizer.from_file(
    str(TOKENIZER_PATH)
)

VOCAB_SIZE = tokenizer.get_vocab_size()

assert VOCAB_SIZE == (
    MODEL_CONFIG["vocab_size"]
), (
    f"Tokenizer vocabulary size {VOCAB_SIZE} does not match "
    f"Architecture-C configuration "
    f"{MODEL_CONFIG['vocab_size']}."
)

VOCAB = tokenizer.get_vocab()

# -------------------------------------------------------------------------
# Load tokenizer metadata.
# -------------------------------------------------------------------------
with open(
    TOKENIZER_METADATA_PATH,
    "r",
    encoding="utf-8",
) as f:
    TOKENIZER_METADATA = json.load(f)

# -------------------------------------------------------------------------
# Load dataset metadata.
# -------------------------------------------------------------------------
with open(
    DATASET_METADATA_PATH,
    "r",
    encoding="utf-8",
) as f:
    DATASET_METADATA = json.load(f)

# -------------------------------------------------------------------------
# Load tokenization manifest.
# -------------------------------------------------------------------------
with open(
    TOKENIZATION_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as f:
    TOKENIZATION_MANIFEST = json.load(f)


# -------------------------------------------------------------------------
# Resolve special-token IDs without assuming that <pad> exists.
#
# Padding is NOT required for Architecture C because training samples have
# fixed sequence lengths and are not padded.
# -------------------------------------------------------------------------
def _resolve_token_id(
    candidates,
    required=True,
    token_metadata=None,
):
    # First try the actual tokenizer vocabulary.
    for token in candidates:
        token_id = VOCAB.get(token)

        if token_id is not None:
            return int(token_id)

    # Then inspect tokenizer metadata recursively for explicit token IDs.
    def _search_metadata(obj):
        if isinstance(obj, dict):
            for key, value in obj.items():
                key_lower = str(key).lower()

                if (
                    any(
                        candidate.lower() in key_lower
                        for candidate in candidates
                    )
                    and isinstance(value, int)
                ):
                    return int(value)

                found = _search_metadata(value)

                if found is not None:
                    return found

        elif isinstance(obj, list):
            for item in obj:
                found = _search_metadata(item)

                if found is not None:
                    return found

        return None

    metadata_id = _search_metadata(
        token_metadata
    )

    if metadata_id is not None:
        return metadata_id

    if required:
        raise RuntimeError(
            f"Required special token not found. "
            f"Tried: {candidates}"
        )

    return None


# -------------------------------------------------------------------------
# The tokenizer may use different conventional spellings.
# -------------------------------------------------------------------------
PAD_ID = _resolve_token_id(
    (
        "<pad>",
        "[PAD]",
        "<PAD>",
    ),
    required=False,
    token_metadata=TOKENIZER_METADATA,
)

UNK_ID = _resolve_token_id(
    (
        "<unk>",
        "[UNK]",
        "<UNK>",
    ),
    required=True,
    token_metadata=TOKENIZER_METADATA,
)

BOS_ID = _resolve_token_id(
    (
        "<bos>",
        "<BOS>",
        "[BOS]",
        "<s>",
        "[CLS]",
    ),
    required=True,
    token_metadata=TOKENIZER_METADATA,
)

EOS_ID = _resolve_token_id(
    (
        "<eos>",
        "<EOS>",
        "[EOS]",
        "</s>",
        "[SEP]",
    ),
    required=True,
    token_metadata=TOKENIZER_METADATA,
)

# -------------------------------------------------------------------------
# Record aggregate corpus statistics when available.
# -------------------------------------------------------------------------
def _metadata_value(
    metadata,
    candidates,
    default=None,
):
    if not isinstance(
        metadata,
        dict,
    ):
        return default

    for key in candidates:
        if key in metadata:
            return metadata[key]

    return default


TOTAL_DOCUMENTS = _metadata_value(
    DATASET_METADATA,
    (
        "total_documents",
        "num_documents",
        "documents",
    ),
)

TOTAL_TRAIN_TOKENS = _metadata_value(
    DATASET_METADATA,
    (
        "total_train_tokens",
        "train_tokens",
        "num_train_tokens",
    ),
)

TOTAL_VALIDATION_TOKENS = _metadata_value(
    DATASET_METADATA,
    (
        "total_validation_tokens",
        "validation_tokens",
        "num_validation_tokens",
        "val_tokens",
    ),
)

# -------------------------------------------------------------------------
# Save runtime tokenizer information.
# -------------------------------------------------------------------------
TOKENIZER_RUNTIME_METADATA = {
    "tokenizer_path": str(TOKENIZER_PATH),
    "vocab_size": int(VOCAB_SIZE),
    "special_token_ids": {
        "pad": (
            None
            if PAD_ID is None
            else int(PAD_ID)
        ),
        "unk": int(UNK_ID),
        "bos": int(BOS_ID),
        "eos": int(EOS_ID),
    },
    "padding_required": False,
    "tokenizer_metadata": TOKENIZER_METADATA,
    "dataset_metadata": DATASET_METADATA,
    "tokenization_manifest": TOKENIZATION_MANIFEST,
}

TOKENIZER_RUNTIME_METADATA_PATH = (
    CONFIGS_DIR
    / "tokenizer_runtime_metadata.json"
)

with open(
    TOKENIZER_RUNTIME_METADATA_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        TOKENIZER_RUNTIME_METADATA,
        f,
        ensure_ascii=False,
        indent=2,
    )

# -------------------------------------------------------------------------
# Runtime summary.
# -------------------------------------------------------------------------
print("✅ Phase-1 tokenizer and metadata loaded.")
print(
    f"Vocabulary size : {VOCAB_SIZE:,}"
)
print(
    f"<pad> ID        : {PAD_ID}"
)
print(
    f"<unk> ID        : {UNK_ID}"
)
print(
    f"BOS ID          : {BOS_ID}"
)
print(
    f"EOS ID          : {EOS_ID}"
)

if PAD_ID is None:
    print(
        "ℹ️ No padding token is present; "
        "this is valid because Architecture-C uses fixed-length sequences."
    )

if TOTAL_DOCUMENTS is not None:
    print(
        f"Documents       : {TOTAL_DOCUMENTS:,}"
    )

if TOTAL_TRAIN_TOKENS is not None:
    print(
        f"Train tokens    : {TOTAL_TRAIN_TOKENS:,}"
    )

if TOTAL_VALIDATION_TOKENS is not None:
    print(
        f"Validation tokens: {TOTAL_VALIDATION_TOKENS:,}"
    )

✅ Phase-1 tokenizer and metadata loaded.
Vocabulary size : 50,000
<pad> ID        : 0
<unk> ID        : 1
BOS ID          : 2
EOS ID          : 3


In [4]:
# Cell 04 — Register the four tokenized legal sources and validate shard metadata
# DECISION: ADD

import json
from pathlib import Path

# -------------------------------------------------------------------------
# Register the four official Phase-1 training sources.
# Architecture C uses the same persistent corpus as Architectures A and B.
# -------------------------------------------------------------------------
SOURCE_CONFIG = {
    "acts": {
        "train_path": TOKENIZED_DIR / "acts_train.bin",
        "val_path": TOKENIZED_DIR / "acts_val.bin",
    },
    "high_courts": {
        "train_path": TOKENIZED_DIR / "high_courts_train.bin",
        "val_path": TOKENIZED_DIR / "high_courts_val.bin",
    },
    "supreme_court": {
        "train_path": TOKENIZED_DIR / "supreme_court_train.bin",
        "val_path": TOKENIZED_DIR / "supreme_court_val.bin",
    },
    "tribunals": {
        "train_path": TOKENIZED_DIR / "tribunals_train.bin",
        "val_path": TOKENIZED_DIR / "tribunals_val.bin",
    },
}

# -------------------------------------------------------------------------
# Validate all shard files and collect their exact token counts from disk.
# The persistent shards contain uint16 token IDs.
# -------------------------------------------------------------------------
SOURCE_METADATA = {}

for _source_name, _paths in SOURCE_CONFIG.items():
    _train_path = Path(
        _paths["train_path"]
    )
    _val_path = Path(
        _paths["val_path"]
    )

    assert _train_path.is_file(), (
        f"Missing training shard: {_train_path}"
    )

    assert _val_path.is_file(), (
        f"Missing validation shard: {_val_path}"
    )

    # uint16 = 2 bytes/token
    assert (
        _train_path.stat().st_size % 2 == 0
    ), (
        f"Training shard size is not divisible by 2: "
        f"{_train_path}"
    )

    assert (
        _val_path.stat().st_size % 2 == 0
    ), (
        f"Validation shard size is not divisible by 2: "
        f"{_val_path}"
    )

    _train_tokens = (
        _train_path.stat().st_size // 2
    )

    _val_tokens = (
        _val_path.stat().st_size // 2
    )

    assert _train_tokens > 0
    assert _val_tokens > 0

    SOURCE_METADATA[_source_name] = {
        "train_path": str(_train_path),
        "validation_path": str(_val_path),
        "train_tokens": int(_train_tokens),
        "validation_tokens": int(_val_tokens),
        "mix_weight": float(
            SOURCE_MIX[_source_name]
        ),
    }

# -------------------------------------------------------------------------
# Verify the configured mixture is valid.
# -------------------------------------------------------------------------
assert set(
    SOURCE_METADATA.keys()
) == set(
    SOURCE_NAMES
)

assert abs(
    sum(
        item["mix_weight"]
        for item in SOURCE_METADATA.values()
    ) - 1.0
) < 1e-8

# -------------------------------------------------------------------------
# Aggregate counts.
# -------------------------------------------------------------------------
TOTAL_REGISTERED_TRAIN_TOKENS = sum(
    item["train_tokens"]
    for item in SOURCE_METADATA.values()
)

TOTAL_REGISTERED_VALIDATION_TOKENS = sum(
    item["validation_tokens"]
    for item in SOURCE_METADATA.values()
)

# -------------------------------------------------------------------------
# Persist source registry.
# -------------------------------------------------------------------------
SOURCE_REGISTRY = {
    "experiment_id": EXPERIMENT_ID,
    "architecture_name": ARCHITECTURE_NAME,
    "sources": SOURCE_METADATA,
    "total_train_tokens": int(
        TOTAL_REGISTERED_TRAIN_TOKENS
    ),
    "total_validation_tokens": int(
        TOTAL_REGISTERED_VALIDATION_TOKENS
    ),
}

SOURCE_REGISTRY_PATH = (
    CONFIGS_DIR / "source_registry.json"
)

with open(
    SOURCE_REGISTRY_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        SOURCE_REGISTRY,
        f,
        ensure_ascii=False,
        indent=2,
    )

# -------------------------------------------------------------------------
# Runtime summary.
# -------------------------------------------------------------------------
print("✅ Official Phase-1 sources registered.")

for _source_name in SOURCE_NAMES:
    _meta = SOURCE_METADATA[
        _source_name
    ]

    print(
        f"{_source_name:15s} | "
        f"train={_meta['train_tokens']:,} | "
        f"val={_meta['validation_tokens']:,} | "
        f"mix={_meta['mix_weight']:.2f}"
    )

print(
    f"\nTotal train tokens : "
    f"{TOTAL_REGISTERED_TRAIN_TOKENS:,}"
)

print(
    f"Total validation tokens: "
    f"{TOTAL_REGISTERED_VALIDATION_TOKENS:,}"
)

print(
    f"Source registry   : "
    f"{SOURCE_REGISTRY_PATH}"
)

✅ Official Phase-1 sources registered.
acts            | train=149,598,391 | val=3,405,295 | mix=0.25
high_courts     | train=2,327,101,667 | val=42,068,902 | mix=0.25
supreme_court   | train=251,833,809 | val=5,313,054 | mix=0.25
tribunals       | train=1,607,073,435 | val=32,846,795 | mix=0.25

Total train tokens : 4,335,607,302
Total validation tokens: 83,634,046
Source registry   : /kaggle/working/architecture_C/configs/source_registry.json


In [5]:
# Cell 05 — Implement Architecture-C: tied-embedding stacked LSTM language model
# DECISION: UPDATE

import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------------------------------------------------------
# Architecture C
#
# This is a genuinely recurrent language model, not a Transformer.
#
# token IDs
#    ↓
# shared token embedding
#    ↓
# stacked LSTM
#    ↓
# hidden-to-embedding projection
#    ↓
# tied vocabulary projection
#    ↓
# next-token logits
#
# The projection allows hidden_size != embedding_dim while still tying the
# output vocabulary matrix to the input embedding matrix.
# -------------------------------------------------------------------------
class DeepLSTMLanguageModel(
    nn.Module
):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_size,
        num_layers,
        dropout,
        tie_embeddings=True,
        use_bias=True,
    ):
        super().__init__()

        self.vocab_size = int(vocab_size)
        self.embedding_dim = int(embedding_dim)
        self.hidden_size = int(hidden_size)
        self.num_layers = int(num_layers)
        self.dropout_probability = float(dropout)
        self.tie_embeddings = bool(tie_embeddings)
        self.use_bias = bool(use_bias)

        # -----------------------------------------------------------------
        # Shared input embedding.
        # -----------------------------------------------------------------
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
        )

        # -----------------------------------------------------------------
        # Stacked recurrent sequence model.
        # -----------------------------------------------------------------
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=(
                dropout
                if num_layers > 1
                else 0.0
            ),
            batch_first=True,
            bias=use_bias,
        )

        # -----------------------------------------------------------------
        # Hidden -> embedding projection.
        #
        # This is identity when hidden_size == embedding_dim and a learned
        # projection otherwise.
        # -----------------------------------------------------------------
        if hidden_size == embedding_dim:
            self.hidden_to_embedding = nn.Identity()
        else:
            self.hidden_to_embedding = nn.Linear(
                hidden_size,
                embedding_dim,
                bias=use_bias,
            )

        # -----------------------------------------------------------------
        # Final normalization is applied in embedding space.
        # -----------------------------------------------------------------
        self.final_norm = nn.LayerNorm(
            embedding_dim
        )

        # -----------------------------------------------------------------
        # Optional vocabulary bias.
        # -----------------------------------------------------------------
        if use_bias:
            self.output_bias = nn.Parameter(
                torch.zeros(vocab_size)
            )
        else:
            self.register_parameter(
                "output_bias",
                None,
            )

        self._initialize_parameters()

    def _initialize_parameters(self):
        nn.init.normal_(
            self.token_embedding.weight,
            mean=0.0,
            std=0.02,
        )

        for name, parameter in self.lstm.named_parameters():
            if "weight_ih" in name:
                nn.init.xavier_uniform_(
                    parameter
                )

            elif "weight_hh" in name:
                nn.init.orthogonal_(
                    parameter
                )

            elif "bias" in name:
                nn.init.zeros_(
                    parameter
                )

                # LSTM gate order:
                # input, forget, cell, output
                hidden = self.hidden_size

                parameter.data[
                    hidden:2 * hidden
                ].fill_(1.0)

        # Initialize optional hidden-to-embedding projection.
        if isinstance(
            self.hidden_to_embedding,
            nn.Linear,
        ):
            nn.init.xavier_uniform_(
                self.hidden_to_embedding.weight
            )

            if (
                self.hidden_to_embedding.bias
                is not None
            ):
                nn.init.zeros_(
                    self.hidden_to_embedding.bias
                )

        nn.init.ones_(
            self.final_norm.weight
        )

        nn.init.zeros_(
            self.final_norm.bias
        )

        if self.output_bias is not None:
            nn.init.zeros_(
                self.output_bias
            )

    def forward(
        self,
        input_ids,
        hidden_state=None,
        return_hidden_state=False,
    ):
        # -------------------------------------------------------------
        # [B, T] -> [B, T, E]
        # -------------------------------------------------------------
        embeddings = self.token_embedding(
            input_ids
        )

        # -------------------------------------------------------------
        # Recurrent processing.
        # -------------------------------------------------------------
        hidden_sequence, hidden_state = (
            self.lstm(
                embeddings,
                hidden_state,
            )
        )

        # -------------------------------------------------------------
        # [B, T, H] -> [B, T, E]
        # -------------------------------------------------------------
        hidden_sequence = (
            self.hidden_to_embedding(
                hidden_sequence
            )
        )

        # -------------------------------------------------------------
        # Normalize in embedding space.
        # -------------------------------------------------------------
        hidden_sequence = self.final_norm(
            hidden_sequence
        )

        # -------------------------------------------------------------
        # Tied vocabulary projection.
        #
        # Input embedding:
        #     [V, E]
        #
        # Hidden representation:
        #     [B, T, E]
        #
        # Result:
        #     [B, T, V]
        # -------------------------------------------------------------
        logits = F.linear(
            hidden_sequence,
            self.token_embedding.weight,
            self.output_bias,
        )

        if return_hidden_state:
            return logits, hidden_state

        return logits

    def init_hidden(
        self,
        batch_size,
        device,
        dtype=None,
    ):
        if dtype is None:
            dtype = self.token_embedding.weight.dtype

        hidden = torch.zeros(
            self.num_layers,
            batch_size,
            self.hidden_size,
            device=device,
            dtype=dtype,
        )

        cell = torch.zeros(
            self.num_layers,
            batch_size,
            self.hidden_size,
            device=device,
            dtype=dtype,
        )

        return hidden, cell


# -------------------------------------------------------------------------
# Instantiate Architecture C.
# -------------------------------------------------------------------------
MODEL = DeepLSTMLanguageModel(
    vocab_size=MODEL_CONFIG["vocab_size"],
    embedding_dim=MODEL_CONFIG["embedding_dim"],
    hidden_size=MODEL_CONFIG["hidden_size"],
    num_layers=MODEL_CONFIG["num_layers"],
    dropout=MODEL_CONFIG["dropout"],
    tie_embeddings=MODEL_CONFIG["tie_embeddings"],
    use_bias=MODEL_CONFIG["use_bias"],
)

# -------------------------------------------------------------------------
# Parameter accounting.
# -------------------------------------------------------------------------
TOTAL_PARAMETERS = sum(
    parameter.numel()
    for parameter in MODEL.parameters()
    if parameter.requires_grad
)

TRAINABLE_PARAMETERS = sum(
    parameter.numel()
    for parameter in MODEL.parameters()
    if parameter.requires_grad
)

TOTAL_PARAMETER_MILLIONS = (
    TOTAL_PARAMETERS / 1_000_000
)

# -------------------------------------------------------------------------
# Production architecture must satisfy the official model-size range.
# FAST mode is only a pipeline-validation model and is intentionally smaller.
# -------------------------------------------------------------------------
if RUN_MODE == "production":
    assert 50_000_000 <= (
        TOTAL_PARAMETERS
    ) <= 300_000_000, (
        f"Architecture-C has "
        f"{TOTAL_PARAMETERS:,} parameters, "
        "outside the required 50M–300M range."
    )

# -------------------------------------------------------------------------
# Device placement.
# -------------------------------------------------------------------------
MODEL = MODEL.to(
    DEVICE
)

# -------------------------------------------------------------------------
# Save model configuration.
# -------------------------------------------------------------------------
MODEL_CONFIG_PATH = (
    CONFIGS_DIR /
    "model_C_config.json"
)

MODEL_CONFIG_METADATA = {
    "architecture_name": ARCHITECTURE_NAME,
    "architecture_version": ARCHITECTURE_VERSION,
    "model_type": "stacked_lstm_language_model",
    "model_config": MODEL_CONFIG,
    "parameter_count": int(
        TOTAL_PARAMETERS
    ),
    "trainable_parameter_count": int(
        TRAINABLE_PARAMETERS
    ),
    "parameter_millions": float(
        TOTAL_PARAMETER_MILLIONS
    ),
    "weight_tying": bool(
        MODEL_CONFIG["tie_embeddings"]
    ),
    "hidden_to_embedding_projection": (
        MODEL_CONFIG["hidden_size"]
        != MODEL_CONFIG["embedding_dim"]
    ),
    "pretrained_weights": False,
}

with open(
    MODEL_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        MODEL_CONFIG_METADATA,
        f,
        ensure_ascii=False,
        indent=2,
    )

# -------------------------------------------------------------------------
# Runtime summary.
# -------------------------------------------------------------------------
print(
    "✅ Architecture-C model instantiated."
)

print(
    f"Model type          : "
    f"Stacked LSTM"
)

print(
    f"Embedding dim       : "
    f"{MODEL_CONFIG['embedding_dim']}"
)

print(
    f"Hidden size         : "
    f"{MODEL_CONFIG['hidden_size']}"
)

print(
    f"LSTM layers         : "
    f"{MODEL_CONFIG['num_layers']}"
)

print(
    f"Context length      : "
    f"{MODEL_CONFIG['context_length']}"
)

print(
    f"Parameters          : "
    f"{TOTAL_PARAMETERS:,} "
    f"({TOTAL_PARAMETER_MILLIONS:.2f}M)"
)

print(
    f"Hidden→embedding    : "
    f"{MODEL_CONFIG['hidden_size']} → "
    f"{MODEL_CONFIG['embedding_dim']}"
)

print(
    f"Weight tying        : "
    f"{MODEL_CONFIG['tie_embeddings']}"
)

print(
    f"Pretrained          : False"
)

✅ Architecture-C model instantiated.
Model type          : Stacked LSTM
Embedding dim       : 1280
Hidden size         : 1280
LSTM layers         : 3
Context length      : 1024
Parameters          : 103,404,880 (103.40M)
Hidden→embedding    : 1280 → 1280
Weight tying        : True
Pretrained          : False


In [6]:
# Cell 06 — Implement memory-mapped LSTM sequence sampling with checkpointable RNG
# DECISION: ADD

import numpy as np
import torch


# -------------------------------------------------------------------------
# Memory-mapped token shard
#
# The .bin files are uint16 token IDs produced by Notebook 1.
# They are never fully loaded into RAM.
# -------------------------------------------------------------------------
class TokenShard:
    def __init__(
        self,
        path,
    ):
        self.path = str(path)

        self.data = np.memmap(
            self.path,
            dtype=np.uint16,
            mode="r",
        )

        assert self.data.ndim == 1
        assert self.data.size > 0

    @property
    def token_count(self):
        return int(
            self.data.size
        )

    def get_block(
        self,
        start,
        length,
    ):
        start = int(start)
        length = int(length)

        assert start >= 0
        assert length > 0
        assert (
            start + length
            <= self.token_count
        )

        return np.asarray(
            self.data[
                start:start + length
            ],
            dtype=np.int64,
        )


# -------------------------------------------------------------------------
# Build all official train/validation shards.
# -------------------------------------------------------------------------
TRAIN_SHARDS = {}
VAL_SHARDS = {}

for _source_name in SOURCE_NAMES:
    TRAIN_SHARDS[_source_name] = (
        TokenShard(
            SOURCE_METADATA[
                _source_name
            ]["train_path"]
        )
    )

    VAL_SHARDS[_source_name] = (
        TokenShard(
            SOURCE_METADATA[
                _source_name
            ]["validation_path"]
        )
    )

# -------------------------------------------------------------------------
# Source sampling probabilities.
# -------------------------------------------------------------------------
SOURCE_NAMES_LIST = list(
    SOURCE_NAMES
)

SOURCE_PROBABILITIES = np.asarray(
    [
        SOURCE_MIX[_source_name]
        for _source_name in SOURCE_NAMES_LIST
    ],
    dtype=np.float64,
)

assert np.isclose(
    SOURCE_PROBABILITIES.sum(),
    1.0,
)

# -------------------------------------------------------------------------
# Checkpointable sequence sampler.
#
# One sample:
#
# tokens[0 : context_length + 1]
#
# input  = tokens[:-1]
# target = tokens[1:]
#
# Thus every position predicts the following token.
# -------------------------------------------------------------------------
class LSTMSequenceSampler:
    def __init__(
        self,
        shards,
        source_names,
        source_probabilities,
        context_length,
        seed,
    ):
        self.shards = shards
        self.source_names = list(
            source_names
        )
        self.source_probabilities = np.asarray(
            source_probabilities,
            dtype=np.float64,
        )
        self.context_length = int(
            context_length
        )

        self.rng = np.random.default_rng(
            int(seed)
        )

        self.minimum_shard_tokens = (
            self.context_length + 1
        )

        for _source_name in self.source_names:
            assert (
                self.shards[
                    _source_name
                ].token_count
                >= self.minimum_shard_tokens
            ), (
                f"Shard {_source_name} is too short "
                f"for context length "
                f"{self.context_length}."
            )

    def sample_one(self):
        # -------------------------------------------------------------
        # Select source according to the configured mixture.
        # -------------------------------------------------------------
        source_index = int(
            self.rng.choice(
                len(self.source_names),
                p=self.source_probabilities,
            )
        )

        source_name = (
            self.source_names[
                source_index
            ]
        )

        shard = self.shards[
            source_name
        ]

        # -------------------------------------------------------------
        # Random contiguous sequence.
        # -------------------------------------------------------------
        max_start = (
            shard.token_count
            - self.context_length
            - 1
        )

        start = int(
            self.rng.integers(
                0,
                max_start + 1,
            )
        )

        block = shard.get_block(
            start,
            self.context_length + 1,
        )

        input_ids = block[:-1]
        target_ids = block[1:]

        return (
            input_ids,
            target_ids,
            source_name,
        )

    def sample_batch(
        self,
        batch_size,
    ):
        batch_size = int(
            batch_size
        )

        assert batch_size > 0

        inputs = np.empty(
            (
                batch_size,
                self.context_length,
            ),
            dtype=np.int64,
        )

        targets = np.empty(
            (
                batch_size,
                self.context_length,
            ),
            dtype=np.int64,
        )

        source_names = []

        for row in range(
            batch_size
        ):
            input_ids, target_ids, source_name = (
                self.sample_one()
            )

            inputs[row] = input_ids
            targets[row] = target_ids
            source_names.append(
                source_name
            )

        return (
            torch.from_numpy(inputs),
            torch.from_numpy(targets),
            source_names,
        )

    def state_dict(self):
        return {
            "context_length": (
                self.context_length
            ),
            "source_names": list(
                self.source_names
            ),
            "source_probabilities": (
                self.source_probabilities.tolist()
            ),
            "rng_state": self.rng.bit_generator.state,
        }

    def load_state_dict(
        self,
        state,
    ):
        assert (
            int(state["context_length"])
            == self.context_length
        )

        assert state[
            "source_names"
        ] == self.source_names

        loaded_probabilities = np.asarray(
            state[
                "source_probabilities"
            ],
            dtype=np.float64,
        )

        assert np.allclose(
            loaded_probabilities,
            self.source_probabilities,
        )

        self.rng = np.random.default_rng()

        self.rng.bit_generator.state = (
            state["rng_state"]
        )


# -------------------------------------------------------------------------
# Create independently checkpointable training and validation samplers.
# -------------------------------------------------------------------------
TRAIN_SAMPLER = LSTMSequenceSampler(
    shards=TRAIN_SHARDS,
    source_names=SOURCE_NAMES_LIST,
    source_probabilities=SOURCE_PROBABILITIES,
    context_length=MODEL_CONFIG[
        "context_length"
    ],
    seed=SEED + 101,
)

VAL_SAMPLER = LSTMSequenceSampler(
    shards=VAL_SHARDS,
    source_names=SOURCE_NAMES_LIST,
    source_probabilities=SOURCE_PROBABILITIES,
    context_length=MODEL_CONFIG[
        "context_length"
    ],
    seed=SEED + 202,
)

# -------------------------------------------------------------------------
# Data-loader style batch utility.
# -------------------------------------------------------------------------
def get_lstm_batch(
    sampler,
    batch_size,
):
    input_ids, target_ids, source_names = (
        sampler.sample_batch(
            batch_size
        )
    )

    input_ids = input_ids.to(
        DEVICE,
        non_blocking=True,
    )

    target_ids = target_ids.to(
        DEVICE,
        non_blocking=True,
    )

    return (
        input_ids,
        target_ids,
        source_names,
    )


# -------------------------------------------------------------------------
# Runtime information.
# -------------------------------------------------------------------------
print(
    "✅ LSTM sequence sampling initialized."
)

print(
    f"Context length : "
    f"{MODEL_CONFIG['context_length']}"
)

print(
    f"Train batch    : "
    f"{TRAINING_CONFIG['micro_batch_size']}"
)

print(
    f"Source mixture : "
    f"{dict(SOURCE_MIX)}"
)

for _source_name in SOURCE_NAMES_LIST:
    print(
        f"{_source_name:15s} | "
        f"train={TRAIN_SHARDS[_source_name].token_count:,} | "
        f"val={VAL_SHARDS[_source_name].token_count:,}"
    )

✅ LSTM sequence sampling initialized.
Context length : 1024
Train batch    : 1
Source mixture : {'acts': 0.25, 'high_courts': 0.25, 'supreme_court': 0.25, 'tribunals': 0.25}
acts            | train=149,598,391 | val=3,405,295
high_courts     | train=2,327,101,667 | val=42,068,902
supreme_court   | train=251,833,809 | val=5,313,054
tribunals       | train=1,607,073,435 | val=32,846,795


In [7]:
# Cell 07 — Configure Architecture-C optimization, mixed precision, and LR schedule
# DECISION: ADD

import contextlib
import torch
import torch.optim as optim

# -------------------------------------------------------------------------
# Optimizer
# -------------------------------------------------------------------------
OPTIMIZER = optim.AdamW(
    MODEL.parameters(),
    lr=TRAINING_CONFIG["learning_rate"],
    betas=(0.9, 0.95),
    weight_decay=TRAINING_CONFIG["weight_decay"],
)

# -------------------------------------------------------------------------
# Cosine learning-rate schedule
#
# The schedule spans the configured maximum training budget. Early stopping
# may terminate before the schedule reaches its final step.
# -------------------------------------------------------------------------
MAX_STEPS = int(
    TRAINING_CONFIG["max_steps"]
)

WARMUP_STEPS = min(
    int(
        TRAINING_CONFIG["warmup_steps"]
    ),
    max(1, MAX_STEPS // 10),
)

BASE_LR = float(
    TRAINING_CONFIG["learning_rate"]
)

MIN_LR = float(
    TRAINING_CONFIG["min_learning_rate"]
)


def get_learning_rate(
    step,
):
    step = int(step)

    if step < WARMUP_STEPS:
        return BASE_LR * (
            (step + 1)
            / WARMUP_STEPS
        )

    if MAX_STEPS <= WARMUP_STEPS:
        return MIN_LR

    progress = (
        (step - WARMUP_STEPS)
        / (MAX_STEPS - WARMUP_STEPS)
    )

    progress = min(
        max(progress, 0.0),
        1.0,
    )

    cosine_value = (
        0.5
        * (
            1.0
            + math.cos(
                math.pi * progress
            )
        )
    )

    return (
        MIN_LR
        + (
            BASE_LR - MIN_LR
        )
        * cosine_value
    )


def set_learning_rate(
    optimizer,
    step,
):
    lr = get_learning_rate(
        step
    )

    for group in optimizer.param_groups:
        group["lr"] = lr

    return lr


# -------------------------------------------------------------------------
# Precision policy
#
# CUDA:
#   Prefer BF16, otherwise FP16 + GradScaler.
#
# MPS/CPU:
#   Keep FP32 for stability and compatibility.
# -------------------------------------------------------------------------
USE_AMP = False
AMP_DTYPE = None
USE_GRAD_SCALER = False

if DEVICE.type == "cuda":
    if torch.cuda.is_bf16_supported():
        USE_AMP = True
        AMP_DTYPE = torch.bfloat16
        USE_GRAD_SCALER = False
    else:
        USE_AMP = True
        AMP_DTYPE = torch.float16
        USE_GRAD_SCALER = True

if USE_GRAD_SCALER:
    SCALER = torch.amp.GradScaler(
        "cuda",
        enabled=True,
    )
else:
    SCALER = torch.amp.GradScaler(
        "cuda",
        enabled=False,
    )


@contextlib.contextmanager
def autocast_context():
    if USE_AMP:
        with torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        ):
            yield
    else:
        yield


# -------------------------------------------------------------------------
# Lightweight device-memory reporting.
# -------------------------------------------------------------------------
def get_device_memory_info():
    if DEVICE.type != "cuda":
        return {
            "allocated_bytes": 0,
            "reserved_bytes": 0,
            "peak_allocated_bytes": 0,
        }

    return {
        "allocated_bytes": int(
            torch.cuda.memory_allocated(
                DEVICE
            )
        ),
        "reserved_bytes": int(
            torch.cuda.memory_reserved(
                DEVICE
            )
        ),
        "peak_allocated_bytes": int(
            torch.cuda.max_memory_allocated(
                DEVICE
            )
        ),
    }


def clear_device_cache():
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    elif DEVICE.type == "mps":
        torch.mps.empty_cache()


# -------------------------------------------------------------------------
# Gradient helper
# -------------------------------------------------------------------------
def clip_gradients(
    model,
    max_norm,
):
    return torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm,
    )


# -------------------------------------------------------------------------
# Persist optimizer/training configuration.
# -------------------------------------------------------------------------
OPTIMIZER_CONFIG = {
    "optimizer": "AdamW",
    "betas": [
        0.9,
        0.95,
    ],
    "weight_decay": float(
        TRAINING_CONFIG["weight_decay"]
    ),
    "base_learning_rate": BASE_LR,
    "minimum_learning_rate": MIN_LR,
    "warmup_steps": WARMUP_STEPS,
    "max_steps": MAX_STEPS,
    "schedule": "warmup_cosine_decay",
    "gradient_clip": float(
        TRAINING_CONFIG["grad_clip"]
    ),
    "mixed_precision": USE_AMP,
    "amp_dtype": (
        None
        if AMP_DTYPE is None
        else str(AMP_DTYPE)
    ),
    "gradient_scaler": USE_GRAD_SCALER,
}

OPTIMIZER_CONFIG_PATH = (
    CONFIGS_DIR / "optimizer_config.json"
)

with open(
    OPTIMIZER_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        OPTIMIZER_CONFIG,
        f,
        ensure_ascii=False,
        indent=2,
    )

# -------------------------------------------------------------------------
# Initial learning rate.
# -------------------------------------------------------------------------
INITIAL_LR = set_learning_rate(
    OPTIMIZER,
    0,
)

# -------------------------------------------------------------------------
# Runtime summary.
# -------------------------------------------------------------------------
print(
    "✅ Architecture-C optimizer configured."
)

print(
    f"Optimizer       : AdamW"
)

print(
    f"Maximum steps   : {MAX_STEPS:,}"
)

print(
    f"Warmup steps    : {WARMUP_STEPS:,}"
)

print(
    f"Base LR         : {BASE_LR:.3e}"
)

print(
    f"Minimum LR      : {MIN_LR:.3e}"
)

print(
    f"Gradient clip   : "
    f"{TRAINING_CONFIG['grad_clip']}"
)

print(
    f"Mixed precision : {USE_AMP}"
)

if AMP_DTYPE is not None:
    print(
        f"AMP dtype       : {AMP_DTYPE}"
    )

print(
    f"Initial LR      : {INITIAL_LR:.3e}"
)

✅ Architecture-C optimizer configured.
Optimizer       : AdamW
Maximum steps   : 12,208
Warmup steps    : 500
Base LR         : 3.000e-04
Minimum LR      : 3.000e-05
Gradient clip   : 1.0
Mixed precision : True
AMP dtype       : torch.bfloat16
Initial LR      : 6.000e-07


In [8]:
# Cell 08 — Build Architecture-C checkpoint, compatibility, and resume infrastructure
# DECISION: ADD

import copy
import json
import hashlib
import random
import os
import tempfile
from pathlib import Path

# -------------------------------------------------------------------------
# Checkpoint paths
# -------------------------------------------------------------------------
LATEST_CHECKPOINT_PATH = (
    CHECKPOINTS_DIR / "latest.pt"
)

BEST_CHECKPOINT_PATH = (
    CHECKPOINTS_DIR / "best.pt"
)

# -------------------------------------------------------------------------
# Stable configuration payload used to prevent incompatible resumes.
#
# A checkpoint may resume only when the architecture, model structure,
# training structure, tokenizer, source mixture, and early-stopping policy
# match the current experiment.
# -------------------------------------------------------------------------
CHECKPOINT_COMPATIBILITY_PAYLOAD = {
    "architecture_name": ARCHITECTURE_NAME,
    "architecture_version": ARCHITECTURE_VERSION,
    "run_mode": RUN_MODE,
    "model_config": copy.deepcopy(
        MODEL_CONFIG
    ),
    "training_structure": {
        "context_length": int(
            MODEL_CONFIG["context_length"]
        ),
        "micro_batch_size": int(
            TRAINING_CONFIG[
                "micro_batch_size"
            ]
        ),
        "gradient_accumulation_steps": int(
            TRAINING_CONFIG[
                "gradient_accumulation_steps"
            ]
        ),
        "train_tokens": int(
            TRAINING_CONFIG[
                "train_tokens"
            ]
        ),
        "validation_tokens": int(
            TRAINING_CONFIG[
                "validation_tokens"
            ]
        ),
        "max_steps": int(
            TRAINING_CONFIG[
                "max_steps"
            ]
        ),
    },
    "optimizer_structure": {
        "optimizer": "AdamW",
        "betas": [
            0.9,
            0.95,
        ],
        "weight_decay": float(
            TRAINING_CONFIG[
                "weight_decay"
            ]
        ),
        "base_learning_rate": float(
            TRAINING_CONFIG[
                "learning_rate"
            ]
        ),
        "minimum_learning_rate": float(
            TRAINING_CONFIG[
                "min_learning_rate"
            ]
        ),
        "warmup_steps": int(
            WARMUP_STEPS
        ),
    },
    "source_names": list(
        SOURCE_NAMES_LIST
    ),
    "source_probabilities": (
        SOURCE_PROBABILITIES.tolist()
    ),
    "tokenizer_path": str(
        TOKENIZER_PATH
    ),
    "tokenizer_vocab_size": int(
        VOCAB_SIZE
    ),
    "special_token_ids": {
        "unk": int(UNK_ID),
        "bos": int(BOS_ID),
        "eos": int(EOS_ID),
        "pad": (
            None
            if PAD_ID is None
            else int(PAD_ID)
        ),
    },
    "early_stopping": copy.deepcopy(
        EARLY_STOPPING_CONFIG
    ),
}

# -------------------------------------------------------------------------
# Generate a stable compatibility hash.
# -------------------------------------------------------------------------
_compatibility_json = json.dumps(
    CHECKPOINT_COMPATIBILITY_PAYLOAD,
    sort_keys=True,
    separators=(",", ":"),
)

CHECKPOINT_COMPATIBILITY_HASH = (
    hashlib.sha256(
        _compatibility_json.encode(
            "utf-8"
        )
    ).hexdigest()
)

# -------------------------------------------------------------------------
# Save/load RNG state.
# -------------------------------------------------------------------------
def get_rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }

    if torch.cuda.is_available():
        state["torch_cuda"] = (
            torch.cuda.get_rng_state_all()
        )

    return state


def restore_rng_state(
    state,
):
    random.setstate(
        state["python"]
    )

    np.random.set_state(
        state["numpy"]
    )

    torch.set_rng_state(
        state["torch"]
    )

    if (
        torch.cuda.is_available()
        and "torch_cuda" in state
    ):
        torch.cuda.set_rng_state_all(
            state["torch_cuda"]
        )


# -------------------------------------------------------------------------
# Atomic checkpoint writer.
# -------------------------------------------------------------------------
def atomic_torch_save(
    payload,
    destination,
):
    destination = Path(
        destination
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = destination.with_suffix(
        destination.suffix + ".tmp"
    )

    torch.save(
        payload,
        temporary_path,
    )

    temporary_path.replace(
        destination
    )


# -------------------------------------------------------------------------
# Build a complete resumable checkpoint.
# -------------------------------------------------------------------------
def build_checkpoint(
    *,
    step,
    tokens_seen,
    best_validation_loss,
    best_validation_step,
    evaluations_without_improvement,
    training_history,
    status,
):
    return {
        "checkpoint_schema_version": 1,
        "experiment_id": EXPERIMENT_ID,
        "architecture_name": ARCHITECTURE_NAME,
        "architecture_version": ARCHITECTURE_VERSION,
        "run_mode": RUN_MODE,
        "compatibility_hash": (
            CHECKPOINT_COMPATIBILITY_HASH
        ),
        "model": MODEL.state_dict(),
        "optimizer": OPTIMIZER.state_dict(),
        "scheduler": {
            "current_step": int(step),
            "learning_rate": float(
                OPTIMIZER.param_groups[0]["lr"]
            ),
        },
        "scaler": SCALER.state_dict(),
        "train_sampler": (
            TRAIN_SAMPLER.state_dict()
        ),
        "validation_sampler": (
            VAL_SAMPLER.state_dict()
        ),
        "rng": get_rng_state(),
        "progress": {
            "optimizer_step": int(step),
            "tokens_seen": int(tokens_seen),
            "best_validation_loss": (
                float(best_validation_loss)
            ),
            "best_validation_step": (
                int(best_validation_step)
            ),
            "evaluations_without_improvement": (
                int(evaluations_without_improvement)
            ),
            "status": status,
        },
        "training_history": copy.deepcopy(
            training_history
        ),
        "configuration": (
            CHECKPOINT_COMPATIBILITY_PAYLOAD
        ),
    }


# -------------------------------------------------------------------------
# Save a checkpoint.
# -------------------------------------------------------------------------
def save_checkpoint(
    path,
    *,
    step,
    tokens_seen,
    best_validation_loss,
    best_validation_step,
    evaluations_without_improvement,
    training_history,
    status,
):
    checkpoint = build_checkpoint(
        step=step,
        tokens_seen=tokens_seen,
        best_validation_loss=best_validation_loss,
        best_validation_step=best_validation_step,
        evaluations_without_improvement=(
            evaluations_without_improvement
        ),
        training_history=training_history,
        status=status,
    )

    atomic_torch_save(
        checkpoint,
        path,
    )


# -------------------------------------------------------------------------
# Check checkpoint compatibility.
# -------------------------------------------------------------------------
def checkpoint_is_compatible(
    checkpoint,
):
    return (
        checkpoint.get(
            "checkpoint_schema_version"
        )
        == 1
        and checkpoint.get(
            "experiment_id"
        )
        == EXPERIMENT_ID
        and checkpoint.get(
            "architecture_name"
        )
        == ARCHITECTURE_NAME
        and checkpoint.get(
            "architecture_version"
        )
        == ARCHITECTURE_VERSION
        and checkpoint.get(
            "run_mode"
        )
        == RUN_MODE
        and checkpoint.get(
            "compatibility_hash"
        )
        == CHECKPOINT_COMPATIBILITY_HASH
    )


# -------------------------------------------------------------------------
# Resume the latest compatible checkpoint.
# -------------------------------------------------------------------------
def load_latest_checkpoint():
    if not LATEST_CHECKPOINT_PATH.is_file():
        return None

    checkpoint = torch.load(
        LATEST_CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=False,
    )

    if not checkpoint_is_compatible(
        checkpoint
    ):
        print(
            "⚠️ Latest checkpoint is incompatible "
            "with the current Architecture-C configuration."
        )
        print(
            "   A fresh training run will be started."
        )
        return None

    MODEL.load_state_dict(
        checkpoint["model"]
    )

    OPTIMIZER.load_state_dict(
        checkpoint["optimizer"]
    )

    # Restore optimizer LR exactly as recorded.
    recorded_lr = checkpoint[
        "scheduler"
    ].get("learning_rate")

    if recorded_lr is not None:
        for group in OPTIMIZER.param_groups:
            group["lr"] = float(
                recorded_lr
            )

    if checkpoint.get("scaler"):
        SCALER.load_state_dict(
            checkpoint["scaler"]
        )

    TRAIN_SAMPLER.load_state_dict(
        checkpoint["train_sampler"]
    )

    VAL_SAMPLER.load_state_dict(
        checkpoint["validation_sampler"]
    )

    restore_rng_state(
        checkpoint["rng"]
    )

    return checkpoint


# -------------------------------------------------------------------------
# Persist the compatibility metadata separately for experiment records.
# -------------------------------------------------------------------------
CHECKPOINT_CONFIG_PATH = (
    CONFIGS_DIR /
    "checkpoint_config.json"
)

CHECKPOINT_CONFIG = {
    "schema_version": 1,
    "compatibility_hash": (
        CHECKPOINT_COMPATIBILITY_HASH
    ),
    "latest_checkpoint": str(
        LATEST_CHECKPOINT_PATH
    ),
    "best_checkpoint": str(
        BEST_CHECKPOINT_PATH
    ),
    "automatic_resume": True,
    "checkpoint_every_steps": int(
        TRAINING_CONFIG[
            "checkpoint_every_steps"
        ]
    ),
    "early_stopping": (
        EARLY_STOPPING_CONFIG
    ),
}

with open(
    CHECKPOINT_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        CHECKPOINT_CONFIG,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(
    "✅ Architecture-C checkpoint infrastructure ready."
)

print(
    f"Compatibility hash : "
    f"{CHECKPOINT_COMPATIBILITY_HASH[:16]}..."
)

print(
    f"Latest checkpoint  : "
    f"{LATEST_CHECKPOINT_PATH}"
)

print(
    f"Best checkpoint    : "
    f"{BEST_CHECKPOINT_PATH}"
)

print(
    "Automatic resume   : True"
)

✅ Architecture-C checkpoint infrastructure ready.
Compatibility hash : b172e6cc0fe2466b...
Latest checkpoint  : /kaggle/working/architecture_C/checkpoints/latest.pt
Best checkpoint    : /kaggle/working/architecture_C/checkpoints/best.pt
Automatic resume   : True


In [9]:
# Cell 08.5 — Restore Architecture-C checkpoint from Phase13
# DECISION: ADD

from pathlib import Path
import shutil
import torch

print("📦 Restoring Architecture-C checkpoint from Phase13...")
print()

INPUT_ROOT = Path("/kaggle/input")

TARGET_ROOT = Path(
    "/kaggle/working/architecture_C"
)

TARGET_CHECKPOINT_DIR = (
    TARGET_ROOT / "checkpoints"
)

TARGET_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# =========================================================================
# FIND CHECKPOINT
# =========================================================================

latest_candidates = sorted(
    [
        p
        for p in INPUT_ROOT.rglob("latest.pt")
        if (
            "architecture_C" in p.parts
            and "checkpoints" in p.parts
        )
    ]
)

if not latest_candidates:

    raise FileNotFoundError(
        "❌ Architecture-C latest.pt was not found "
        "inside /kaggle/input.\n"
        "Make sure Phase13 is attached to this notebook."
    )

print("✅ Found checkpoint candidate(s):")

for p in latest_candidates:
    print(f"   {p}")

SOURCE_LATEST = (
    latest_candidates[0]
)

SOURCE_CHECKPOINT_DIR = (
    SOURCE_LATEST.parent
)

print()
print(
    f"Using checkpoint:\n"
    f"   {SOURCE_LATEST}"
)

# =========================================================================
# INSPECT CHECKPOINT BEFORE COPYING
# =========================================================================

print()
print("🔍 Inspecting saved training state...")

checkpoint = torch.load(
    SOURCE_LATEST,
    map_location="cpu",
    weights_only=False,
)

assert isinstance(
    checkpoint,
    dict,
), "❌ Checkpoint is not a dictionary."

print(
    f"Architecture       : "
    f"{checkpoint.get('architecture_name')}"
)

print(
    f"Architecture ver.  : "
    f"{checkpoint.get('architecture_version')}"
)

print(
    f"Run mode           : "
    f"{checkpoint.get('run_mode')}"
)

print(
    f"Experiment ID      : "
    f"{checkpoint.get('experiment_id')}"
)

print(
    f"Compatibility hash : "
    f"{str(checkpoint.get('compatibility_hash', ''))[:16]}..."
)

# =========================================================================
# CHECK TRAINING PROGRESS
# =========================================================================

progress = checkpoint.get(
    "progress"
)

if not isinstance(
    progress,
    dict,
):
    raise RuntimeError(
        "❌ Checkpoint does not contain a valid "
        "'progress' dictionary."
    )

saved_step = int(
    progress.get(
        "optimizer_step",
        0,
    )
)

saved_tokens = int(
    progress.get(
        "tokens_seen",
        0,
    )
)

saved_best_loss = float(
    progress.get(
        "best_validation_loss",
        float("inf"),
    )
)

saved_best_step = int(
    progress.get(
        "best_validation_step",
        0,
    )
)

print()
print(
    f"Saved optimizer step : "
    f"{saved_step:,}"
)

print(
    f"Saved tokens seen    : "
    f"{saved_tokens:,}"
)

print(
    f"Saved best val loss  : "
    f"{saved_best_loss:.6f}"
)

print(
    f"Saved best step      : "
    f"{saved_best_step:,}"
)

# =========================================================================
# STRICT COMPATIBILITY CHECK
# =========================================================================

assert (
    checkpoint.get("architecture_name")
    == ARCHITECTURE_NAME
), (
    "❌ Architecture mismatch."
)

assert (
    checkpoint.get("architecture_version")
    == ARCHITECTURE_VERSION
), (
    "❌ Architecture version mismatch."
)

assert (
    checkpoint.get("experiment_id")
    == EXPERIMENT_ID
), (
    "❌ Experiment ID mismatch."
)

assert (
    checkpoint.get("run_mode")
    == RUN_MODE
), (
    "❌ RUN_MODE mismatch. "
    "Use production."
)

assert (
    checkpoint.get("compatibility_hash")
    == CHECKPOINT_COMPATIBILITY_HASH
), (
    "❌ Compatibility hash mismatch.\n"
    "The current Architecture-C configuration "
    "does not exactly match the saved checkpoint."
)

required_keys = {
    "model",
    "optimizer",
    "scheduler",
    "scaler",
    "train_sampler",
    "validation_sampler",
    "rng",
    "progress",
    "training_history",
}

missing_keys = (
    required_keys
    - set(checkpoint.keys())
)

assert not missing_keys, (
    f"❌ Checkpoint is missing required state: "
    f"{sorted(missing_keys)}"
)

print()
print(
    "✅ Checkpoint is fully compatible."
)

del checkpoint

# =========================================================================
# COPY LATEST CHECKPOINT
# =========================================================================

TARGET_LATEST = (
    TARGET_CHECKPOINT_DIR
    / "latest.pt"
)

print()
print("⏳ Copying latest.pt...")

shutil.copy2(
    SOURCE_LATEST,
    TARGET_LATEST,
)

print(
    "✅ latest.pt restored."
)

# =========================================================================
# COPY BEST CHECKPOINT
# =========================================================================

SOURCE_BEST = (
    SOURCE_CHECKPOINT_DIR
    / "best.pt"
)

TARGET_BEST = (
    TARGET_CHECKPOINT_DIR
    / "best.pt"
)

if SOURCE_BEST.exists():

    print(
        "⏳ Copying best.pt..."
    )

    shutil.copy2(
        SOURCE_BEST,
        TARGET_BEST,
    )

    print(
        "✅ best.pt restored."
    )

else:

    print(
        "⚠️ best.pt not found. "
        "Resume can still use latest.pt."
    )

# =========================================================================
# FINAL VERIFICATION
# =========================================================================

print()
print("=" * 72)
print("✅ ARCHITECTURE-C RESTORE COMPLETE")
print("=" * 72)

print(
    f"latest.pt exists : "
    f"{TARGET_LATEST.exists()}"
)

print(
    f"latest.pt size   : "
    f"{TARGET_LATEST.stat().st_size / (1024**3):.2f} GB"
)

print(
    f"best.pt exists   : "
    f"{TARGET_BEST.exists()}"
)

print(
    f"Resume step      : "
    f"{saved_step:,}"
)

print(
    f"Resume tokens    : "
    f"{saved_tokens:,}"
)

print("=" * 72)

📦 Restoring Architecture-C checkpoint from Phase13...

✅ Found checkpoint candidate(s):
   /kaggle/input/datasets/rohitbatra2004/phase133/architecture_C/checkpoints/latest.pt

Using checkpoint:
   /kaggle/input/datasets/rohitbatra2004/phase133/architecture_C/checkpoints/latest.pt

🔍 Inspecting saved training state...
Architecture       : Architecture_C_Deep_LSTM_Language_Model
Architecture ver.  : C_v1
Run mode           : production
Experiment ID      : CS781_Phase1_C_v1_production
Compatibility hash : b172e6cc0fe2466b...

Saved optimizer step : 12,208
Saved tokens seen    : 100,007,936
Saved best val loss  : 3.870509
Saved best step      : 12,000

✅ Checkpoint is fully compatible.

⏳ Copying latest.pt...
✅ latest.pt restored.
⏳ Copying best.pt...
✅ best.pt restored.

✅ ARCHITECTURE-C RESTORE COMPLETE
latest.pt exists : True
latest.pt size   : 1.16 GB
best.pt exists   : True
Resume step      : 12,208
Resume tokens    : 100,007,936


In [10]:
# Cell 09 — Implement Architecture-C validation, training, and automatic early stopping
# DECISION: UPDATE

import copy
import time
import math
import torch
import torch.nn.functional as F

# -------------------------------------------------------------------------
# Language-model loss
# -------------------------------------------------------------------------
def compute_language_model_loss(
    logits,
    targets,
):
    batch_size = logits.size(0)
    sequence_length = logits.size(1)
    vocabulary_size = logits.size(2)

    logits = logits.reshape(
        batch_size * sequence_length,
        vocabulary_size,
    )

    targets = targets.reshape(
        batch_size * sequence_length
    )

    return F.cross_entropy(
        logits,
        targets,
    )


# -------------------------------------------------------------------------
# Validation evaluation
# -------------------------------------------------------------------------
@torch.no_grad()
def evaluate_architecture_C(
    eval_batches=None,
):
    if eval_batches is None:
        eval_batches = int(
            TRAINING_CONFIG["eval_batches"]
        )

    MODEL.eval()

    total_loss = 0.0
    total_batches = 0
    total_tokens = 0

    for _ in range(eval_batches):
        input_ids, target_ids, _ = (
            get_lstm_batch(
                VAL_SAMPLER,
                TRAINING_CONFIG[
                    "micro_batch_size"
                ],
            )
        )

        with autocast_context():
            logits = MODEL(
                input_ids
            )

            loss = compute_language_model_loss(
                logits,
                target_ids,
            )

        total_loss += float(
            loss.detach().float().item()
        )

        total_batches += 1
        total_tokens += int(
            target_ids.numel()
        )

    validation_loss = (
        total_loss
        / max(total_batches, 1)
    )

    validation_perplexity = math.exp(
        min(validation_loss, 20.0)
    )

    return {
        "loss": float(validation_loss),
        "perplexity": float(
            validation_perplexity
        ),
        "tokens": int(total_tokens),
        "batches": int(total_batches),
    }


# -------------------------------------------------------------------------
# Training state
# -------------------------------------------------------------------------
TRAINING_HISTORY = {
    "train": [],
    "validation": [],
}

CURRENT_STEP = 0
TOKENS_SEEN = 0

BEST_VALIDATION_LOSS = float("inf")
BEST_VALIDATION_STEP = 0

EVALUATIONS_WITHOUT_IMPROVEMENT = 0

LAST_VALIDATION_RESULT = None
TRAINING_STATUS = "not_started"


# -------------------------------------------------------------------------
# Automatic resume
# -------------------------------------------------------------------------
_RESUME_CHECKPOINT = (
    load_latest_checkpoint()
)

if _RESUME_CHECKPOINT is not None:
    _progress = _RESUME_CHECKPOINT[
        "progress"
    ]

    CURRENT_STEP = int(
        _progress["optimizer_step"]
    )

    TOKENS_SEEN = int(
        _progress["tokens_seen"]
    )

    BEST_VALIDATION_LOSS = float(
        _progress["best_validation_loss"]
    )

    BEST_VALIDATION_STEP = int(
        _progress["best_validation_step"]
    )

    EVALUATIONS_WITHOUT_IMPROVEMENT = int(
        _progress[
            "evaluations_without_improvement"
        ]
    )

    TRAINING_HISTORY = copy.deepcopy(
        _RESUME_CHECKPOINT.get(
            "training_history",
            TRAINING_HISTORY,
        )
    )

    TRAINING_STATUS = str(
        _progress.get(
            "status",
            "resumed",
        )
    )

    print(
        "♻️ Resumed Architecture-C training."
    )

    print(
        f"   Step          : "
        f"{CURRENT_STEP:,}"
    )

    print(
        f"   Tokens seen   : "
        f"{TOKENS_SEEN:,}"
    )

    print(
        f"   Best val loss : "
        f"{BEST_VALIDATION_LOSS:.6f}"
    )

    print(
        f"   Best step     : "
        f"{BEST_VALIDATION_STEP:,}"
    )

else:
    print(
        "🆕 No compatible checkpoint found; "
        "starting Architecture-C from scratch."
    )


# -------------------------------------------------------------------------
# Checkpoint helper
# -------------------------------------------------------------------------
def _save_training_checkpoint(
    path,
    *,
    step,
    tokens_seen,
    best_validation_loss,
    best_validation_step,
    evaluations_without_improvement,
    status,
):
    save_checkpoint(
        path,
        step=step,
        tokens_seen=tokens_seen,
        best_validation_loss=(
            best_validation_loss
        ),
        best_validation_step=(
            best_validation_step
        ),
        evaluations_without_improvement=(
            evaluations_without_improvement
        ),
        training_history=TRAINING_HISTORY,
        status=status,
    )


# -------------------------------------------------------------------------
# Main training loop
# -------------------------------------------------------------------------
def train_architecture_C():
    global CURRENT_STEP
    global TOKENS_SEEN
    global BEST_VALIDATION_LOSS
    global BEST_VALIDATION_STEP
    global EVALUATIONS_WITHOUT_IMPROVEMENT
    global LAST_VALIDATION_RESULT
    global TRAINING_STATUS

    MAX_STEPS = int(
        TRAINING_CONFIG["max_steps"]
    )

    GRAD_ACCUM = int(
        TRAINING_CONFIG[
            "gradient_accumulation_steps"
        ]
    )

    MICRO_BATCH = int(
        TRAINING_CONFIG[
            "micro_batch_size"
        ]
    )

    CONTEXT_LENGTH = int(
        MODEL_CONFIG["context_length"]
    )

    EVAL_EVERY = int(
        TRAINING_CONFIG[
            "eval_every_steps"
        ]
    )

    CHECKPOINT_EVERY = int(
        TRAINING_CONFIG[
            "checkpoint_every_steps"
        ]
    )

    LOG_EVERY = int(
        TRAINING_CONFIG[
            "log_every_steps"
        ]
    )

    MIN_STEPS = int(
        EARLY_STOPPING_CONFIG[
            "minimum_steps_before_stop"
        ]
    )

    PATIENCE = int(
        EARLY_STOPPING_CONFIG[
            "patience_evaluations"
        ]
    )

    MIN_DELTA = float(
        EARLY_STOPPING_CONFIG[
            "min_delta"
        ]
    )

    MODEL.train()

    OPTIMIZER.zero_grad(
        set_to_none=True
    )

    if CURRENT_STEP >= MAX_STEPS:
        TRAINING_STATUS = (
            "max_steps_reached"
        )

        # Make sure a final state exists.
        _save_training_checkpoint(
            LATEST_CHECKPOINT_PATH,
            step=CURRENT_STEP,
            tokens_seen=TOKENS_SEEN,
            best_validation_loss=(
                BEST_VALIDATION_LOSS
            ),
            best_validation_step=(
                BEST_VALIDATION_STEP
            ),
            evaluations_without_improvement=(
                EVALUATIONS_WITHOUT_IMPROVEMENT
            ),
            status=TRAINING_STATUS,
        )

        print(
            "✅ Training already reached max_steps."
        )

        return

    print(
        "\n🚀 Architecture-C training started."
    )

    print(
        f"Maximum steps       : {MAX_STEPS:,}"
    )

    print(
        f"Effective tokens/step: "
        f"{MICRO_BATCH * GRAD_ACCUM * CONTEXT_LENGTH:,}"
    )

    print(
        f"Early stopping      : "
        f"{EARLY_STOPPING_CONFIG['enabled']}"
    )

    print(
        f"Patience            : "
        f"{PATIENCE} validations"
    )

    print(
        f"Minimum delta       : "
        f"{MIN_DELTA}"
    )

    start_time = time.time()

    while CURRENT_STEP < MAX_STEPS:
        MODEL.train()

        optimizer_step_loss = 0.0

        for _micro_step in range(
            GRAD_ACCUM
        ):
            input_ids, target_ids, _ = (
                get_lstm_batch(
                    TRAIN_SAMPLER,
                    MICRO_BATCH,
                )
            )

            with autocast_context():
                logits = MODEL(
                    input_ids
                )

                loss = compute_language_model_loss(
                    logits,
                    target_ids,
                )

                scaled_loss = (
                    loss / GRAD_ACCUM
                )

            if USE_GRAD_SCALER:
                SCALER.scale(
                    scaled_loss
                ).backward()
            else:
                scaled_loss.backward()

            optimizer_step_loss += float(
                loss.detach().float().item()
            )

            TOKENS_SEEN += int(
                target_ids.numel()
            )

            del input_ids
            del target_ids
            del logits
            del loss
            del scaled_loss

        # -------------------------------------------------------------
        # Gradient clipping and optimizer step
        # -------------------------------------------------------------
        if USE_GRAD_SCALER:
            SCALER.unscale_(
                OPTIMIZER
            )

        gradient_norm = clip_gradients(
            MODEL,
            TRAINING_CONFIG[
                "grad_clip"
            ],
        )

        if USE_GRAD_SCALER:
            SCALER.step(
                OPTIMIZER
            )
            SCALER.update()
        else:
            OPTIMIZER.step()

        OPTIMIZER.zero_grad(
            set_to_none=True
        )

        # -------------------------------------------------------------
        # Advance step and LR
        # -------------------------------------------------------------
        CURRENT_STEP += 1

        CURRENT_LR = set_learning_rate(
            OPTIMIZER,
            CURRENT_STEP,
        )

        mean_train_loss = (
            optimizer_step_loss
            / GRAD_ACCUM
        )

        TRAINING_HISTORY[
            "train"
        ].append(
            {
                "step": int(
                    CURRENT_STEP
                ),
                "tokens_seen": int(
                    TOKENS_SEEN
                ),
                "loss": float(
                    mean_train_loss
                ),
                "learning_rate": float(
                    CURRENT_LR
                ),
                "gradient_norm": float(
                    gradient_norm.detach()
                    .float()
                    .item()
                    if torch.is_tensor(
                        gradient_norm
                    )
                    else gradient_norm
                ),
            }
        )

        # -------------------------------------------------------------
        # Training log
        # -------------------------------------------------------------
        if (
            CURRENT_STEP % LOG_EVERY == 0
            or CURRENT_STEP == 1
        ):
            elapsed = (
                time.time()
                - start_time
            )

            print(
                f"Step "
                f"{CURRENT_STEP:>6,}/"
                f"{MAX_STEPS:,} | "
                f"train_loss="
                f"{mean_train_loss:.4f} | "
                f"lr="
                f"{CURRENT_LR:.3e} | "
                f"tokens="
                f"{TOKENS_SEEN:,} | "
                f"elapsed="
                f"{elapsed/60:.1f}m"
            )

        # -------------------------------------------------------------
        # Validation
        # -------------------------------------------------------------
        should_evaluate = (
            CURRENT_STEP % EVAL_EVERY == 0
            or CURRENT_STEP == MAX_STEPS
        )

        if should_evaluate:
            validation_result = (
                evaluate_architecture_C()
            )

            LAST_VALIDATION_RESULT = (
                validation_result
            )

            validation_loss = float(
                validation_result["loss"]
            )

            validation_ppl = float(
                validation_result["perplexity"]
            )

            TRAINING_HISTORY[
                "validation"
            ].append(
                {
                    "step": int(
                        CURRENT_STEP
                    ),
                    "tokens_seen": int(
                        TOKENS_SEEN
                    ),
                    "loss": validation_loss,
                    "perplexity": validation_ppl,
                    "tokens": int(
                        validation_result[
                            "tokens"
                        ]
                    ),
                }
            )

            print(
                f"           val_loss="
                f"{validation_loss:.4f} | "
                f"val_ppl="
                f"{validation_ppl:.4f}"
            )

            improvement = (
                BEST_VALIDATION_LOSS
                - validation_loss
            )

            is_new_best = (
                improvement
                >= MIN_DELTA
            )

            if is_new_best:
                BEST_VALIDATION_LOSS = (
                    validation_loss
                )

                BEST_VALIDATION_STEP = (
                    CURRENT_STEP
                )

                EVALUATIONS_WITHOUT_IMPROVEMENT = 0

                _save_training_checkpoint(
                    BEST_CHECKPOINT_PATH,
                    step=CURRENT_STEP,
                    tokens_seen=TOKENS_SEEN,
                    best_validation_loss=(
                        BEST_VALIDATION_LOSS
                    ),
                    best_validation_step=(
                        BEST_VALIDATION_STEP
                    ),
                    evaluations_without_improvement=0,
                    status="best",
                )

                print(
                    "           ⭐ New best checkpoint saved."
                )

            else:
                EVALUATIONS_WITHOUT_IMPROVEMENT += 1

                print(
                    "           "
                    f"No meaningful improvement "
                    f"({EVALUATIONS_WITHOUT_IMPROVEMENT}/"
                    f"{PATIENCE})."
                )

            # ---------------------------------------------------------
            # Always save the latest checkpoint.
            # ---------------------------------------------------------
            _save_training_checkpoint(
                LATEST_CHECKPOINT_PATH,
                step=CURRENT_STEP,
                tokens_seen=TOKENS_SEEN,
                best_validation_loss=(
                    BEST_VALIDATION_LOSS
                ),
                best_validation_step=(
                    BEST_VALIDATION_STEP
                ),
                evaluations_without_improvement=(
                    EVALUATIONS_WITHOUT_IMPROVEMENT
                ),
                status="running",
            )

            print(
                "           💾 Latest checkpoint saved."
            )

            # ---------------------------------------------------------
            # Early stopping
            # ---------------------------------------------------------
            if (
                EARLY_STOPPING_CONFIG[
                    "enabled"
                ]
                and CURRENT_STEP >= MIN_STEPS
                and EVALUATIONS_WITHOUT_IMPROVEMENT
                >= PATIENCE
            ):
                TRAINING_STATUS = (
                    "early_stopped"
                )

                _save_training_checkpoint(
                    LATEST_CHECKPOINT_PATH,
                    step=CURRENT_STEP,
                    tokens_seen=TOKENS_SEEN,
                    best_validation_loss=(
                        BEST_VALIDATION_LOSS
                    ),
                    best_validation_step=(
                        BEST_VALIDATION_STEP
                    ),
                    evaluations_without_improvement=(
                        EVALUATIONS_WITHOUT_IMPROVEMENT
                    ),
                    status=TRAINING_STATUS,
                )

                print(
                    "\n🛑 Early stopping triggered."
                )

                break

    # ---------------------------------------------------------------------
    # Normal completion at maximum budget.
    # ---------------------------------------------------------------------
    if CURRENT_STEP >= MAX_STEPS:
        TRAINING_STATUS = (
            "max_steps_reached"
        )

        _save_training_checkpoint(
            LATEST_CHECKPOINT_PATH,
            step=CURRENT_STEP,
            tokens_seen=TOKENS_SEEN,
            best_validation_loss=(
                BEST_VALIDATION_LOSS
            ),
            best_validation_step=(
                BEST_VALIDATION_STEP
            ),
            evaluations_without_improvement=(
                EVALUATIONS_WITHOUT_IMPROVEMENT
            ),
            status=TRAINING_STATUS,
        )

    # ---------------------------------------------------------------------
    # Final training summary.
    # ---------------------------------------------------------------------
    TRAINING_SUMMARY = {
        "experiment_id": EXPERIMENT_ID,
        "architecture_name": ARCHITECTURE_NAME,
        "architecture_version": ARCHITECTURE_VERSION,
        "run_mode": RUN_MODE,
        "status": TRAINING_STATUS,
        "optimizer_steps": int(
            CURRENT_STEP
        ),
        "tokens_seen": int(
            TOKENS_SEEN
        ),
        "max_steps": int(
            MAX_STEPS
        ),
        "best_validation_loss": float(
            BEST_VALIDATION_LOSS
        ),
        "best_validation_perplexity": float(
            math.exp(
                min(
                    BEST_VALIDATION_LOSS,
                    20.0,
                )
            )
        ),
        "best_validation_step": int(
            BEST_VALIDATION_STEP
        ),
        "evaluations_without_improvement": int(
            EVALUATIONS_WITHOUT_IMPROVEMENT
        ),
        "early_stopping": (
            EARLY_STOPPING_CONFIG
        ),
        "training_history": (
            TRAINING_HISTORY
        ),
    }

    TRAINING_SUMMARY_PATH = (
        METRICS_DIR /
        "training_summary_C.json"
    )

    with open(
        TRAINING_SUMMARY_PATH,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            TRAINING_SUMMARY,
            f,
            ensure_ascii=False,
            indent=2,
        )

    print(
        "\n✅ Architecture-C training finished."
    )

    print(
        f"Status              : "
        f"{TRAINING_STATUS}"
    )

    print(
        f"Steps completed     : "
        f"{CURRENT_STEP:,}"
    )

    print(
        f"Tokens seen         : "
        f"{TOKENS_SEEN:,}"
    )

    print(
        f"Best validation loss: "
        f"{BEST_VALIDATION_LOSS:.6f}"
    )

    print(
        f"Best validation PPL : "
        f"{math.exp(min(BEST_VALIDATION_LOSS, 20.0)):.4f}"
    )

    print(
        f"Best checkpoint step: "
        f"{BEST_VALIDATION_STEP:,}"
    )

    print(
        f"Training summary    : "
        f"{TRAINING_SUMMARY_PATH}"
    )


print(
    "✅ Architecture-C training engine ready."
)

print(
    f"Maximum optimizer steps: "
    f"{TRAINING_CONFIG['max_steps']:,}"
)

print(
    f"Early stopping enabled  : "
    f"{EARLY_STOPPING_CONFIG['enabled']}"
)

print(
    f"Minimum steps            : "
    f"{EARLY_STOPPING_CONFIG['minimum_steps_before_stop']:,}"
)

print(
    f"Patience                 : "
    f"{EARLY_STOPPING_CONFIG['patience_evaluations']}"
)

♻️ Resumed Architecture-C training.
   Step          : 12,208
   Tokens seen   : 100,007,936
   Best val loss : 3.870509
   Best step     : 12,000
✅ Architecture-C training engine ready.
Maximum optimizer steps: 12,208
Early stopping enabled  : True
Minimum steps            : 1,000
Patience                 : 4


In [11]:
# Cell 10 — Execute Architecture-C training with automatic early stopping/resume
# DECISION: UPDATE

from pathlib import Path
import json
import math

# -------------------------------------------------------------------------
# Training summary path
# -------------------------------------------------------------------------
TRAINING_SUMMARY_PATH = (
    METRICS_DIR /
    "training_summary_C.json"
)

# -------------------------------------------------------------------------
# Run the training engine.
#
# FAST:
#   tiny pipeline-validation run.
#
# PRODUCTION:
#   production-scale LSTM with a maximum token budget and automatic
#   validation-based early stopping.
#
# The training engine automatically resumes only from a compatible
# checkpoint. FAST and PRODUCTION checkpoints are intentionally treated as
# different experiments.
# -------------------------------------------------------------------------
print(
    f"🚀 Starting Architecture-C "
    f"{RUN_MODE.upper()} training..."
)

train_architecture_C()

# -------------------------------------------------------------------------
# If training returned immediately because max_steps had already been
# reached, make sure the summary still exists.
# -------------------------------------------------------------------------
if not TRAINING_SUMMARY_PATH.is_file():

    if CURRENT_STEP >= int(
        TRAINING_CONFIG["max_steps"]
    ):
        TRAINING_STATUS = (
            "max_steps_reached"
        )

    TRAINING_SUMMARY = {
        "experiment_id": EXPERIMENT_ID,
        "architecture_name": ARCHITECTURE_NAME,
        "architecture_version": ARCHITECTURE_VERSION,
        "run_mode": RUN_MODE,
        "status": TRAINING_STATUS,
        "optimizer_steps": int(
            CURRENT_STEP
        ),
        "tokens_seen": int(
            TOKENS_SEEN
        ),
        "max_steps": int(
            TRAINING_CONFIG["max_steps"]
        ),
        "best_validation_loss": float(
            BEST_VALIDATION_LOSS
        ),
        "best_validation_perplexity": float(
            math.exp(
                min(
                    BEST_VALIDATION_LOSS,
                    20.0,
                )
            )
        ),
        "best_validation_step": int(
            BEST_VALIDATION_STEP
        ),
        "evaluations_without_improvement": int(
            EVALUATIONS_WITHOUT_IMPROVEMENT
        ),
        "early_stopping": copy.deepcopy(
            EARLY_STOPPING_CONFIG
        ),
        "training_history": copy.deepcopy(
            TRAINING_HISTORY
        ),
    }

    with open(
        TRAINING_SUMMARY_PATH,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            TRAINING_SUMMARY,
            f,
            ensure_ascii=False,
            indent=2,
        )

# -------------------------------------------------------------------------
# Required artifact checks.
# -------------------------------------------------------------------------
assert CURRENT_STEP > 0, (
    "Architecture-C completed zero optimizer steps."
)

assert BEST_VALIDATION_STEP > 0, (
    "Architecture-C did not produce a valid "
    "best-validation checkpoint."
)

assert BEST_CHECKPOINT_PATH.is_file(), (
    f"Best checkpoint missing: "
    f"{BEST_CHECKPOINT_PATH}"
)

assert LATEST_CHECKPOINT_PATH.is_file(), (
    f"Latest checkpoint missing: "
    f"{LATEST_CHECKPOINT_PATH}"
)

assert TRAINING_SUMMARY_PATH.is_file(), (
    f"Training summary missing: "
    f"{TRAINING_SUMMARY_PATH}"
)

# -------------------------------------------------------------------------
# Final runtime summary.
# -------------------------------------------------------------------------
print(
    "\n✅ Architecture-C training cell completed."
)

print(
    f"Run mode             : "
    f"{RUN_MODE}"
)

print(
    f"Training status      : "
    f"{TRAINING_STATUS}"
)

print(
    f"Steps completed      : "
    f"{CURRENT_STEP:,}"
)

print(
    f"Tokens seen          : "
    f"{TOKENS_SEEN:,}"
)

print(
    f"Maximum steps        : "
    f"{TRAINING_CONFIG['max_steps']:,}"
)

print(
    f"Best validation loss : "
    f"{BEST_VALIDATION_LOSS:.6f}"
)

print(
    f"Best validation PPL  : "
    f"{math.exp(min(BEST_VALIDATION_LOSS, 20.0)):.4f}"
)

print(
    f"Best checkpoint step : "
    f"{BEST_VALIDATION_STEP:,}"
)

print(
    f"Early stopping       : "
    f"{EARLY_STOPPING_CONFIG['enabled']}"
)

print(
    f"Best checkpoint      : "
    f"{BEST_CHECKPOINT_PATH}"
)

print(
    f"Latest checkpoint    : "
    f"{LATEST_CHECKPOINT_PATH}"
)

print(
    f"Training summary     : "
    f"{TRAINING_SUMMARY_PATH}"
)

🚀 Starting Architecture-C PRODUCTION training...
✅ Training already reached max_steps.

✅ Architecture-C training cell completed.
Run mode             : production
Training status      : max_steps_reached
Steps completed      : 12,208
Tokens seen          : 100,007,936
Maximum steps        : 12,208
Best validation loss : 3.870509
Best validation PPL  : 47.9668
Best checkpoint step : 12,000
Early stopping       : True
Best checkpoint      : /kaggle/working/architecture_C/checkpoints/best.pt
Latest checkpoint    : /kaggle/working/architecture_C/checkpoints/latest.pt
Training summary     : /kaggle/working/architecture_C/metrics/training_summary_C.json


In [12]:
# Cell 11 — Select the best Architecture-C checkpoint and save final model artifacts
# DECISION: ADD

import copy
import json
import math
import torch

# -------------------------------------------------------------------------
# Best checkpoint must exist after the FAST pipeline validation.
# -------------------------------------------------------------------------
assert BEST_CHECKPOINT_PATH.is_file(), (
    f"Best Architecture-C checkpoint not found: "
    f"{BEST_CHECKPOINT_PATH}"
)

BEST_CHECKPOINT = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

assert checkpoint_is_compatible(
    BEST_CHECKPOINT
), (
    "Best Architecture-C checkpoint is incompatible "
    "with the current configuration."
)

# -------------------------------------------------------------------------
# Restore best validation checkpoint into the model.
# -------------------------------------------------------------------------
MODEL.load_state_dict(
    BEST_CHECKPOINT["model"]
)

MODEL = MODEL.to(
    DEVICE
)

MODEL.eval()

# -------------------------------------------------------------------------
# Recover best-checkpoint information.
# -------------------------------------------------------------------------
BEST_CHECKPOINT_STEP = int(
    BEST_CHECKPOINT["progress"][
        "best_validation_step"
    ]
)

BEST_CHECKPOINT_TOKENS = int(
    BEST_CHECKPOINT["progress"][
        "tokens_seen"
    ]
)

BEST_VALIDATION_LOSS = float(
    BEST_CHECKPOINT["progress"][
        "best_validation_loss"
    ]
)

BEST_VALIDATION_PERPLEXITY = math.exp(
    min(
        BEST_VALIDATION_LOSS,
        20.0,
    )
)

# -------------------------------------------------------------------------
# Final model artifact paths.
# -------------------------------------------------------------------------
FINAL_MODEL_PATH = (
    NOTEBOOK_DIR /
    "model_C_final.pt"
)

FINAL_STATE_DICT_PATH = (
    NOTEBOOK_DIR /
    "model_C_state_dict.pt"
)

FINAL_METADATA_PATH = (
    NOTEBOOK_DIR /
    "model_C_metadata.json"
)

# -------------------------------------------------------------------------
# Build the final model artifact.
# -------------------------------------------------------------------------
FINAL_MODEL_ARTIFACT = {
    "artifact_type": (
        "CS781_Phase1_Legal_Language_Model"
    ),
    "architecture_name": (
        ARCHITECTURE_NAME
    ),
    "architecture_version": (
        ARCHITECTURE_VERSION
    ),
    "experiment_id": EXPERIMENT_ID,
    "model_type": (
        "stacked_lstm_language_model"
    ),
    "model_config": copy.deepcopy(
        MODEL_CONFIG
    ),
    "parameter_count": int(
        TOTAL_PARAMETERS
    ),
    "state_dict": MODEL.state_dict(),
    "tokenizer_path": str(
        TOKENIZER_PATH
    ),
    "checkpoint_used": str(
        BEST_CHECKPOINT_PATH
    ),
    "training_step": int(
        BEST_CHECKPOINT_STEP
    ),
    "tokens_seen": int(
        BEST_CHECKPOINT_TOKENS
    ),
    "best_validation_loss": float(
        BEST_VALIDATION_LOSS
    ),
    "best_validation_perplexity": float(
        BEST_VALIDATION_PERPLEXITY
    ),
}

# -------------------------------------------------------------------------
# Atomic model save.
# -------------------------------------------------------------------------
atomic_torch_save(
    FINAL_MODEL_ARTIFACT,
    FINAL_MODEL_PATH,
)

atomic_torch_save(
    MODEL.state_dict(),
    FINAL_STATE_DICT_PATH,
)

# -------------------------------------------------------------------------
# Metadata-only artifact.
# -------------------------------------------------------------------------
FINAL_MODEL_METADATA = {
    "artifact_type": (
        "CS781_Phase1_Legal_Language_Model"
    ),
    "architecture_name": (
        ARCHITECTURE_NAME
    ),
    "architecture_version": (
        ARCHITECTURE_VERSION
    ),
    "experiment_id": EXPERIMENT_ID,
    "model_type": (
        "stacked_lstm_language_model"
    ),
    "parameter_count": int(
        TOTAL_PARAMETERS
    ),
    "parameter_millions": float(
        TOTAL_PARAMETERS / 1_000_000
    ),
    "model_config": copy.deepcopy(
        MODEL_CONFIG
    ),
    "checkpoint_used": str(
        BEST_CHECKPOINT_PATH
    ),
    "training_step": int(
        BEST_CHECKPOINT_STEP
    ),
    "tokens_seen": int(
        BEST_CHECKPOINT_TOKENS
    ),
    "best_validation_loss": float(
        BEST_VALIDATION_LOSS
    ),
    "best_validation_perplexity": float(
        BEST_VALIDATION_PERPLEXITY
    ),
    "early_stopping": copy.deepcopy(
        EARLY_STOPPING_CONFIG
    ),
    "tokenizer_path": str(
        TOKENIZER_PATH
    ),
    "pretrained_weights": False,
}

with open(
    FINAL_METADATA_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        FINAL_MODEL_METADATA,
        f,
        ensure_ascii=False,
        indent=2,
    )

# -------------------------------------------------------------------------
# Save selected-checkpoint metadata.
# -------------------------------------------------------------------------
SELECTED_CHECKPOINT_PATH = (
    METRICS_DIR /
    "selected_checkpoint_C.json"
)

SELECTED_CHECKPOINT_METADATA = {
    "experiment_id": EXPERIMENT_ID,
    "architecture_name": ARCHITECTURE_NAME,
    "selected_checkpoint": str(
        BEST_CHECKPOINT_PATH
    ),
    "training_step": int(
        BEST_CHECKPOINT_STEP
    ),
    "tokens_seen": int(
        BEST_CHECKPOINT_TOKENS
    ),
    "validation_loss": float(
        BEST_VALIDATION_LOSS
    ),
    "validation_perplexity": float(
        BEST_VALIDATION_PERPLEXITY
    ),
    "early_stopping_status": (
        TRAINING_STATUS
    ),
}

with open(
    SELECTED_CHECKPOINT_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        SELECTED_CHECKPOINT_METADATA,
        f,
        ensure_ascii=False,
        indent=2,
    )

# -------------------------------------------------------------------------
# Final checks.
# -------------------------------------------------------------------------
assert FINAL_MODEL_PATH.is_file()
assert FINAL_STATE_DICT_PATH.is_file()
assert FINAL_METADATA_PATH.is_file()
assert SELECTED_CHECKPOINT_PATH.is_file()

if RUN_MODE == "production":
    assert 50_000_000 <= (
        TOTAL_PARAMETERS
    ) <= 300_000_000, (
        f"Production Architecture-C parameter count "
        f"{TOTAL_PARAMETERS:,} is outside 50M–300M."
    )

print(
    "✅ Best Architecture-C checkpoint selected."
)

print(
    f"Parameters          : "
    f"{TOTAL_PARAMETERS:,} "
    f"({TOTAL_PARAMETERS / 1_000_000:.2f}M)"
)

print(
    f"Best checkpoint step: "
    f"{BEST_CHECKPOINT_STEP:,}"
)

print(
    f"Tokens seen         : "
    f"{BEST_CHECKPOINT_TOKENS:,}"
)

print(
    f"Best validation loss: "
    f"{BEST_VALIDATION_LOSS:.6f}"
)

print(
    f"Best validation PPL : "
    f"{BEST_VALIDATION_PERPLEXITY:.4f}"
)

print(
    f"Final model         : "
    f"{FINAL_MODEL_PATH}"
)

print(
    f"State dict          : "
    f"{FINAL_STATE_DICT_PATH}"
)

print(
    f"Metadata            : "
    f"{FINAL_METADATA_PATH}"
)

print(
    f"Selection record    : "
    f"{SELECTED_CHECKPOINT_PATH}"
)

✅ Best Architecture-C checkpoint selected.
Parameters          : 103,404,880 (103.40M)
Best checkpoint step: 12,000
Tokens seen         : 98,304,000
Best validation loss: 3.870509
Best validation PPL : 47.9668
Final model         : /kaggle/working/architecture_C/model_C_final.pt
State dict          : /kaggle/working/architecture_C/model_C_state_dict.pt
Metadata            : /kaggle/working/architecture_C/model_C_metadata.json
Selection record    : /kaggle/working/architecture_C/metrics/selected_checkpoint_C.json


In [13]:
# Cell 12 — Configure Architecture-C generation and inference settings
# DECISION: ADD

# -------------------------------------------------------------------------
# Generation configuration
#
# FAST mode:
#   2 official prompts
#   8 words per continuation
#
# PRODUCTION mode:
#   all 1,000 official prompts
#   exactly 128 words per continuation
#
# max_new_tokens is only a safety ceiling. The actual stopping condition is
# the configured target word count.
# -------------------------------------------------------------------------
if RUN_MODE == "fast":
    GENERATION_CONFIG = {
        "test_prompt_limit": 2,
        "target_words": 8,
        "max_new_tokens": 128,
        "temperature": 0.10,
        "top_p": 0.95,
        "top_k": 0,
        "repetition_penalty": 1.0,
    }

else:
    GENERATION_CONFIG = {
        "test_prompt_limit": None,
        "target_words": 128,
        "max_new_tokens": 1536,
        "temperature": 0.10,
        "top_p": 0.95,
        "top_k": 0,
        "repetition_penalty": 1.0,
    }

# -------------------------------------------------------------------------
# Validate generation configuration.
# -------------------------------------------------------------------------
assert (
    GENERATION_CONFIG["target_words"] > 0
)

assert (
    GENERATION_CONFIG["max_new_tokens"]
    >= GENERATION_CONFIG["target_words"]
)

assert (
    0.0 < GENERATION_CONFIG["temperature"]
)

assert (
    0.0 < GENERATION_CONFIG["top_p"]
    <= 1.0
)

assert (
    GENERATION_CONFIG["top_k"] >= 0
)

assert (
    GENERATION_CONFIG["repetition_penalty"]
    >= 1.0
)

# -------------------------------------------------------------------------
# Persist generation configuration.
# -------------------------------------------------------------------------
GENERATION_CONFIG_PATH = (
    CONFIGS_DIR /
    "generation_config_C.json"
)

GENERATION_CONFIG_METADATA = {
    "experiment_id": EXPERIMENT_ID,
    "architecture_name": ARCHITECTURE_NAME,
    "architecture_version": ARCHITECTURE_VERSION,
    "run_mode": RUN_MODE,
    "generation": GENERATION_CONFIG,
}

with open(
    GENERATION_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        GENERATION_CONFIG_METADATA,
        f,
        ensure_ascii=False,
        indent=2,
    )

# -------------------------------------------------------------------------
# Runtime summary.
# -------------------------------------------------------------------------
print(
    "✅ Architecture-C generation configured."
)

print(
    f"Prompt limit       : "
    f"{GENERATION_CONFIG['test_prompt_limit']}"
)

print(
    f"Target words       : "
    f"{GENERATION_CONFIG['target_words']}"
)

print(
    f"Max new tokens     : "
    f"{GENERATION_CONFIG['max_new_tokens']}"
)

print(
    f"Temperature        : "
    f"{GENERATION_CONFIG['temperature']}"
)

print(
    f"Top-p              : "
    f"{GENERATION_CONFIG['top_p']}"
)

print(
    f"Top-k              : "
    f"{GENERATION_CONFIG['top_k']}"
)

print(
    f"Repetition penalty : "
    f"{GENERATION_CONFIG['repetition_penalty']}"
)

print(
    f"Config saved       : "
    f"{GENERATION_CONFIG_PATH}"
)

✅ Architecture-C generation configured.
Prompt limit       : None
Target words       : 128
Max new tokens     : 1536
Temperature        : 0.1
Top-p              : 0.95
Top-k              : 0
Repetition penalty : 1.0
Config saved       : /kaggle/working/architecture_C/configs/generation_config_C.json


In [14]:
# =========================================================================
# Cell 13 — FINAL Architecture-C efficient autoregressive generation
# DECISION: REPLACE
#
# IMPORTANT:
#   - NO retraining
#   - NO model architecture change
#   - NO checkpoint change
#   - NO tokenizer change
#   - Compatible with the existing trained Architecture-C model
#
# Improvements over previous version:
#   1. Keeps the efficient LSTM hidden/cell state cache.
#   2. Detects stalled whitespace-word generation.
#   3. Uses model-logit-based word-boundary rescue.
#   4. Uses a larger internal safety budget without changing Cell 12.
#   5. Handles pathological sampling distributions more safely.
#   6. Still returns EXACTLY target_words words.
# =========================================================================

import math
import torch
import torch.nn.functional as F


# -------------------------------------------------------------------------
# Sampling helper
# -------------------------------------------------------------------------
def top_p_filtering_C(
    logits,
    top_p=0.95,
):
    top_p = float(top_p)

    if top_p >= 1.0:
        return logits

    sorted_logits, sorted_indices = torch.sort(
        logits,
        descending=True,
    )

    sorted_probs = torch.softmax(
        sorted_logits,
        dim=-1,
    )

    cumulative_probs = torch.cumsum(
        sorted_probs,
        dim=-1,
    )

    remove_mask = (
        cumulative_probs > top_p
    )

    # Keep the first token above the threshold.
    remove_mask[1:] = (
        remove_mask[:-1].clone()
    )

    remove_mask[0] = False

    sorted_logits = sorted_logits.masked_fill(
        remove_mask,
        float("-inf"),
    )

    filtered_logits = torch.full_like(
        logits,
        float("-inf"),
    )

    filtered_logits.scatter_(
        -1,
        sorted_indices,
        sorted_logits,
    )

    return filtered_logits


# -------------------------------------------------------------------------
# Repetition penalty
# -------------------------------------------------------------------------
def apply_repetition_penalty_C(
    logits,
    generated_ids,
    penalty,
):
    penalty = float(penalty)

    if (
        penalty <= 1.0
        or not generated_ids
    ):
        return logits

    adjusted = logits.clone()

    unique_ids = set(
        int(token_id)
        for token_id in generated_ids
    )

    for token_id in unique_ids:

        if not (
            0 <= token_id < adjusted.size(-1)
        ):
            continue

        value = adjusted[
            :,
            token_id,
        ]

        adjusted[
            :,
            token_id,
        ] = torch.where(
            value < 0,
            value * penalty,
            value / penalty,
        )

    return adjusted


# -------------------------------------------------------------------------
# Standard next-token sampling
# -------------------------------------------------------------------------
def select_next_token_C(
    logits,
    temperature,
    top_p,
    top_k,
):
    temperature = max(
        float(temperature),
        1e-5,
    )

    logits = (
        logits / temperature
    )

    # -------------------------------------------------------------
    # Top-k
    # -------------------------------------------------------------
    if (
        top_k is not None
        and int(top_k) > 0
    ):

        k = min(
            int(top_k),
            logits.size(-1),
        )

        values, _ = torch.topk(
            logits,
            k=k,
            dim=-1,
        )

        threshold = values[
            ...,
            -1,
            None,
        ]

        logits = logits.masked_fill(
            logits < threshold,
            float("-inf"),
        )

    # -------------------------------------------------------------
    # Top-p
    # -------------------------------------------------------------
    logits = top_p_filtering_C(
        logits,
        float(top_p),
    )

    probs = torch.softmax(
        logits,
        dim=-1,
    )

    # -------------------------------------------------------------
    # Safety fallback
    # -------------------------------------------------------------
    if (
        not torch.isfinite(probs).all()
        or float(
            probs.sum().item()
        ) <= 0.0
    ):

        return int(
            torch.argmax(
                logits,
                dim=-1,
            )[0].item()
        )

    return int(
        torch.multinomial(
            probs[0],
            num_samples=1,
        ).item()
    )


# -------------------------------------------------------------------------
# Word counter
# -------------------------------------------------------------------------
def count_words_C(
    text,
):
    return len(
        str(text)
        .strip()
        .split()
    )


# Keep compatibility with later cells.
count_words = count_words_C


# -------------------------------------------------------------------------
# Truncate to exact word count
# -------------------------------------------------------------------------
def truncate_to_word_limit_C(
    text,
    target_words,
):
    words = (
        str(text)
        .strip()
        .split()
    )

    return " ".join(
        words[
            :int(target_words)
        ]
    )


# -------------------------------------------------------------------------
# Resolve Architecture-C context length.
# -------------------------------------------------------------------------
LSTM_CONTEXT_LENGTH = int(
    MODEL_CONFIG[
        "context_length"
    ]
)


# -------------------------------------------------------------------------
# Special tokens that must not be generated.
# -------------------------------------------------------------------------
SPECIAL_TOKEN_IDS_C = {
    int(UNK_ID),
    int(BOS_ID),
    int(EOS_ID),
}

if PAD_ID is not None:
    SPECIAL_TOKEN_IDS_C.add(
        int(PAD_ID)
    )


# =========================================================================
# WORD-BOUNDARY RESCUE
# =========================================================================
def find_word_advancing_token_C(
    logits,
    generated_ids,
    current_word_count,
    candidate_k=1024,
    suffix_window=128,
):
    """
    Choose a high-logit token that actually advances the number of
    whitespace-separated words.

    IMPORTANT:
        The token is still selected directly from the model logits.
        No synthetic text is introduced.

    The hidden state is NOT modified here.
    The selected token is passed through the real LSTM immediately
    afterward by the main generation loop.
    """

    candidate_logits = (
        logits
        .float()
        .clone()
    )

    # -------------------------------------------------------------
    # Never select structural tokens.
    # -------------------------------------------------------------
    for token_id in SPECIAL_TOKEN_IDS_C:

        if (
            0 <= token_id
            < candidate_logits.numel()
        ):

            candidate_logits[
                token_id
            ] = -float("inf")

    candidate_k = min(
        int(candidate_k),
        candidate_logits.numel(),
    )

    if candidate_k <= 0:
        return None

    candidate_values, candidate_indices = (
        torch.topk(
            candidate_logits,
            k=candidate_k,
        )
    )

    # -------------------------------------------------------------
    # Only decode a short recent suffix for candidate evaluation.
    #
    # This keeps rescue much cheaper than repeatedly decoding the
    # complete 1,536-token continuation.
    # -------------------------------------------------------------
    recent_ids = (
        generated_ids[
            -int(suffix_window):
        ]
    )

    try:
        base_suffix_text = tokenizer.decode(
            recent_ids,
            skip_special_tokens=True,
        )
    except Exception:
        base_suffix_text = ""

    base_suffix_word_count = (
        count_words_C(
            base_suffix_text
        )
    )

    best_token = None
    best_score = -float("inf")

    for score, token_tensor in zip(
        candidate_values.tolist(),
        candidate_indices.tolist(),
    ):

        score = float(score)

        if not math.isfinite(score):
            continue

        token_id = int(
            token_tensor
        )

        candidate_ids = (
            recent_ids
            + [token_id]
        )

        try:
            candidate_text = (
                tokenizer.decode(
                    candidate_ids,
                    skip_special_tokens=True,
                )
            )
        except Exception:
            continue

        candidate_word_count = (
            count_words_C(
                candidate_text
            )
        )

        # ---------------------------------------------------------
        # Primary condition:
        # candidate must create a new whitespace-separated word.
        # ---------------------------------------------------------
        if (
            candidate_word_count
            > base_suffix_word_count
        ):

            if score > best_score:

                best_score = score
                best_token = token_id

    return best_token


# =========================================================================
# FINAL EFFICIENT ARCHITECTURE-C GENERATOR
# =========================================================================
@torch.inference_mode()
def generate_architecture_C_continuation(
    prompt,
    target_words=None,
    max_new_tokens=None,
    temperature=None,
    top_p=None,
    top_k=None,
    repetition_penalty=None,
):
    """
    Generate an exact number of whitespace-separated continuation words.

    Efficient implementation:
        prompt is processed once,
        then LSTM hidden/cell state is carried forward.

    Recovery:
        if word generation stalls, inspect high-logit candidates and
        select one that advances the whitespace word count.
    """

    generation_cfg = (
        GENERATION_CONFIG
    )

    # -------------------------------------------------------------
    # Resolve configuration defaults.
    # -------------------------------------------------------------
    if target_words is None:
        target_words = int(
            generation_cfg[
                "target_words"
            ]
        )

    if max_new_tokens is None:
        max_new_tokens = int(
            generation_cfg[
                "max_new_tokens"
            ]
        )

    if temperature is None:
        temperature = float(
            generation_cfg[
                "temperature"
            ]
        )

    if top_p is None:
        top_p = float(
            generation_cfg[
                "top_p"
            ]
        )

    if top_k is None:
        top_k = int(
            generation_cfg[
                "top_k"
            ]
        )

    if repetition_penalty is None:
        repetition_penalty = float(
            generation_cfg[
                "repetition_penalty"
            ]
        )

    target_words = int(
        target_words
    )

    max_new_tokens = int(
        max_new_tokens
    )

    temperature = float(
        temperature
    )

    top_p = float(
        top_p
    )

    top_k = int(
        top_k
    )

    repetition_penalty = float(
        repetition_penalty
    )

    assert target_words > 0
    assert max_new_tokens > 0

    prompt = str(
        prompt
    ).strip()

    assert prompt

    MODEL.eval()

    # =============================================================
    # TOKENIZE PROMPT
    # =============================================================
    prompt_ids = tokenizer.encode(
        prompt
    ).ids

    assert len(prompt_ids) > 0, (
        "Prompt produced zero tokenizer IDs."
    )

    # =============================================================
    # CONTEXT WINDOW
    # =============================================================
    if (
        len(prompt_ids)
        > LSTM_CONTEXT_LENGTH
    ):

        if RUN_MODE == "fast":

            prompt_ids = (
                prompt_ids[
                    -LSTM_CONTEXT_LENGTH:
                ]
            )

        else:

            raise ValueError(
                "Production Architecture-C prompt "
                "exceeds the configured context length.\n"
                f"Prompt tokens : {len(prompt_ids)}\n"
                f"Context       : {LSTM_CONTEXT_LENGTH}"
            )

    # =============================================================
    # PROMPT FORWARD PASS
    #
    # The prompt is processed ONCE.
    # =============================================================
    prompt_tensor = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=DEVICE,
    )

    prompt_logits, hidden_state = (
        MODEL(
            prompt_tensor,
            return_hidden_state=True,
        )
    )

    next_logits = (
        prompt_logits[
            :,
            -1,
            :
        ]
        .float()
    )

    # =============================================================
    # GENERATED TOKENS ONLY
    # =============================================================
    generated_ids = []

    decoded_text = ""

    # Number of consecutive generated tokens that failed to
    # increase the whitespace-separated word count.
    stalled_tokens = 0

    last_word_count = 0

    # =============================================================
    # INTERNAL SAFETY BUDGET
    #
    # Cell 12 remains untouched.
    #
    # We allow additional inference tokens when necessary, because
    # the model may need more than the configured ceiling to reach
    # the exact word count.
    # =============================================================
    safety_budget = max(
        max_new_tokens,
        target_words * 16,
    )

    safety_budget = min(
        safety_budget,
        4096,
    )

    # =============================================================
    # AUTOREGRESSIVE LOOP
    # =============================================================
    for step in range(
        safety_budget
    ):

        # ---------------------------------------------------------
        # Apply repetition penalty.
        # ---------------------------------------------------------
        candidate_logits = (
            apply_repetition_penalty_C(
                next_logits,
                generated_ids,
                repetition_penalty,
            )
        )

        # ---------------------------------------------------------
        # Block structural tokens.
        # ---------------------------------------------------------
        for token_id in SPECIAL_TOKEN_IDS_C:

            if (
                0 <= token_id
                < candidate_logits.size(-1)
            ):

                candidate_logits[
                    :,
                    token_id
                ] = -float("inf")

        # ---------------------------------------------------------
        # Current word count.
        # ---------------------------------------------------------
        current_word_count = (
            count_words_C(
                decoded_text
            )
        )

        # ---------------------------------------------------------
        # Word-boundary rescue.
        #
        # Trigger when generation has stalled for several tokens.
        # ---------------------------------------------------------
        use_rescue = (
            stalled_tokens >= 8
            and current_word_count
            < target_words
        )

        if use_rescue:

            rescue_token = (
                find_word_advancing_token_C(
                    candidate_logits[0],
                    generated_ids,
                    current_word_count,
                    candidate_k=1024,
                    suffix_window=128,
                )
            )

            if rescue_token is not None:

                next_token_id = int(
                    rescue_token
                )

            else:

                next_token_id = (
                    select_next_token_C(
                        candidate_logits,
                        temperature=temperature,
                        top_p=top_p,
                        top_k=top_k,
                    )
                )

        else:

            next_token_id = (
                select_next_token_C(
                    candidate_logits,
                    temperature=temperature,
                    top_p=top_p,
                    top_k=top_k,
                )
            )

        # ---------------------------------------------------------
        # Append generated token.
        # ---------------------------------------------------------
        generated_ids.append(
            next_token_id
        )

        # ---------------------------------------------------------
        # Decode continuation only.
        # ---------------------------------------------------------
        decoded_text = (
            tokenizer.decode(
                generated_ids,
                skip_special_tokens=True,
            )
            .strip()
        )

        updated_word_count = (
            count_words_C(
                decoded_text
            )
        )

        # ---------------------------------------------------------
        # Track whether this token advanced word count.
        # ---------------------------------------------------------
        if (
            updated_word_count
            > last_word_count
        ):

            stalled_tokens = 0

        else:

            stalled_tokens += 1

        last_word_count = (
            updated_word_count
        )

        # ---------------------------------------------------------
        # Exact target reached.
        # ---------------------------------------------------------
        if (
            updated_word_count
            >= target_words
        ):

            break

        # ---------------------------------------------------------
        # Carry recurrent state forward.
        #
        # Only the NEW token is processed.
        # ---------------------------------------------------------
        next_input = torch.tensor(
            [[next_token_id]],
            dtype=torch.long,
            device=DEVICE,
        )

        next_logits, hidden_state = (
            MODEL(
                next_input,
                hidden_state=hidden_state,
                return_hidden_state=True,
            )
        )

        next_logits = (
            next_logits[
                :,
                -1,
                :
            ]
            .float()
        )

        # ---------------------------------------------------------
        # Block structural tokens for next iteration.
        # ---------------------------------------------------------
        for token_id in SPECIAL_TOKEN_IDS_C:

            if (
                0 <= token_id
                < next_logits.size(-1)
            ):

                next_logits[
                    :,
                    token_id
                ] = -float("inf")

    # =============================================================
    # FINAL EXACT-WORD NORMALIZATION
    # =============================================================
    continuation = (
        truncate_to_word_limit_C(
            decoded_text,
            target_words,
        )
    )

    final_word_count = (
        count_words_C(
            continuation
        )
    )

    # =============================================================
    # FINAL ASSERTION
    # =============================================================
    assert (
        final_word_count
        == target_words
    ), (
        "\n"
        "❌ Architecture-C still could not reach the "
        "required exact word count.\n\n"
        f"Target words       : {target_words}\n"
        f"Produced words     : {final_word_count}\n"
        f"Generated tokens   : {len(generated_ids)}\n"
        f"Configured max tok : {max_new_tokens}\n"
        f"Internal safety cap: {safety_budget}\n"
        f"Final stalled      : {stalled_tokens}\n"
        f"Context length     : {LSTM_CONTEXT_LENGTH}\n"
    )

    assert continuation.strip()

    return continuation


# =========================================================================
# Compatibility aliases
# =========================================================================
generate_continuation_C = (
    generate_architecture_C_continuation
)

generate_continuation = (
    generate_architecture_C_continuation
)

generate_text = (
    generate_architecture_C_continuation
)


# =========================================================================
# SUMMARY
# =========================================================================
print()
print(
    "✅ FINAL Architecture-C generation helper ready."
)

print(
    f"Context length       : "
    f"{LSTM_CONTEXT_LENGTH}"
)

print(
    f"Target words         : "
    f"{GENERATION_CONFIG['target_words']}"
)

print(
    f"Configured max tokens: "
    f"{GENERATION_CONFIG['max_new_tokens']}"
)

_INTERNAL_SAFETY_CAP = max(
    int(
        GENERATION_CONFIG[
            "max_new_tokens"
        ]
    ),
    int(
        GENERATION_CONFIG[
            "target_words"
        ]
    ) * 16,
)

print(
    f"Internal safety cap  : "
    f"{_INTERNAL_SAFETY_CAP}"
)

print(
    f"Temperature          : "
    f"{GENERATION_CONFIG['temperature']}"
)

print(
    f"Top-p                : "
    f"{GENERATION_CONFIG['top_p']}"
)

print(
    f"Top-k                : "
    f"{GENERATION_CONFIG['top_k']}"
)

print(
    f"Repetition penalty   : "
    f"{GENERATION_CONFIG['repetition_penalty']}"
)

print(
    f"Rescue mechanism     : ENABLED"
)

print(
    f"Rescue trigger       : 8 stalled tokens"
)

print(
    f"Rescue candidates    : 1024"
)

print(
    "LSTM state caching   : ENABLED"
)

print(
    "✅ No retraining required."
)


✅ FINAL Architecture-C generation helper ready.
Context length       : 1024
Target words         : 128
Configured max tokens: 1536
Internal safety cap  : 2048
Temperature          : 0.1
Top-p                : 0.95
Top-k                : 0
Repetition penalty   : 1.0
Rescue mechanism     : ENABLED
Rescue trigger       : 8 stalled tokens
Rescue candidates    : 1024
LSTM state caching   : ENABLED
✅ No retraining required.


In [15]:
# Cell 14 — Discover and load the official Phase-1 test set
# DECISION: UPDATE

import json
import shutil
from pathlib import Path

# -------------------------------------------------------------------------
# Official Hugging Face repository
#
# The course README identifies the Phase-1 dataset as:
#   Exploration-Lab/CS-781-Capstone
# -------------------------------------------------------------------------
HF_REPO_ID = (
    "Exploration-Lab/CS-781-Capstone"
)

# -------------------------------------------------------------------------
# Search local Kaggle storage first.
# -------------------------------------------------------------------------
TEST_JSON_PATH = None

LOCAL_SEARCH_ROOTS = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
]

LOCAL_CANDIDATES = []

for _root in LOCAL_SEARCH_ROOTS:
    if not _root.exists():
        continue

    try:
        LOCAL_CANDIDATES.extend(
            _root.rglob("test.json")
        )

        LOCAL_CANDIDATES.extend(
            _root.rglob("Phase1- Test.json")
        )

        LOCAL_CANDIDATES.extend(
            _root.rglob("Phase1-Test.json")
        )

    except Exception:
        pass

# Prefer paths associated with Phase 1.
_PHASE1_LOCAL = [
    _path
    for _path in LOCAL_CANDIDATES
    if (
        _path.is_file()
        and (
            "phase_1_pretraining" in str(
                _path
            ).lower()
            or "phase1" in str(
                _path
            ).lower()
        )
    )
]

if _PHASE1_LOCAL:
    TEST_JSON_PATH = sorted(
        set(_PHASE1_LOCAL),
        key=lambda p: len(str(p)),
    )[0]

elif LOCAL_CANDIDATES:
    TEST_JSON_PATH = sorted(
        set(LOCAL_CANDIDATES),
        key=lambda p: len(str(p)),
    )[0]

# -------------------------------------------------------------------------
# Discover the CURRENT official test file on Hugging Face.
#
# We do not guess the final filename. Instead, inspect all repository files
# and look for a JSON file associated with Phase 1 test data.
# -------------------------------------------------------------------------
if TEST_JSON_PATH is None:

    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi
    from huggingface_hub import hf_hub_download

    _secrets = UserSecretsClient()

    _hf_token = _secrets.get_secret(
        "HF"
    )

    assert _hf_token, (
        "Kaggle secret 'HF_TOKEN' was not found."
    )

    _hf_api = HfApi(
        token=_hf_token
    )

    print(
        f"🔎 Discovering files in {HF_REPO_ID}..."
    )

    _repo_files = _hf_api.list_repo_files(
        repo_id=HF_REPO_ID,
        repo_type="dataset",
    )

    # ---------------------------------------------------------------------
    # Broad Phase-1 test-file discovery.
    # ---------------------------------------------------------------------
    _test_candidates = []

    for _file in _repo_files:
        _normalized = (
            _file
            .replace("\\", "/")
            .lower()
        )

        _basename = (
            Path(_normalized).name
        )

        is_json = (
            _normalized.endswith(".json")
        )

        is_phase1 = (
            "phase_1_pretraining"
            in _normalized
            or "phase1" in _normalized
            or "phase-1" in _normalized
        )

        looks_like_test = (
            "test" in _basename
        )

        if (
            is_json
            and is_phase1
            and looks_like_test
        ):
            _test_candidates.append(
                _file
            )

    assert _test_candidates, (
        "Could not discover a Phase-1 test JSON in "
        f"{HF_REPO_ID}."
    )

    # Prefer the exact path already established for this project.
    def _candidate_rank(
        path
    ):
        _normalized = (
            path
            .replace("\\", "/")
            .lower()
        )

        _basename = Path(
            _normalized
        ).name

        if (
            _normalized
            == "phase_1_pretraining/test/test.json"
        ):
            return 0

        if _basename == "test.json":
            return 1

        if (
            "phase1- test" in _basename
            or "phase1-test" in _basename
        ):
            return 2

        return 3

    _test_candidates = sorted(
        _test_candidates,
        key=_candidate_rank,
    )

    DISCOVERED_TEST_REPO_PATH = (
        _test_candidates[0]
    )

    print(
        "✅ Discovered official test file:"
    )
    print(
        f"   {DISCOVERED_TEST_REPO_PATH}"
    )

    _downloaded_path = hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=(
            DISCOVERED_TEST_REPO_PATH
        ),
        repo_type="dataset",
        token=_hf_token,
        local_dir=str(
            TEST_DATA_DIR
        ),
    )

    TEST_JSON_PATH = Path(
        _downloaded_path
    )

assert TEST_JSON_PATH is not None
assert TEST_JSON_PATH.is_file(), (
    f"Official test file not found: "
    f"{TEST_JSON_PATH}"
)

print(
    f"📄 Official test file: "
    f"{TEST_JSON_PATH}"
)

# -------------------------------------------------------------------------
# Copy the exact official test file into Architecture-C's own directory.
# -------------------------------------------------------------------------
C_TEST_TARGET = (
    TEST_DATA_DIR
    / "Phase_1_Pretraining"
    / "Test"
    / "test.json"
)

C_TEST_TARGET.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if (
    TEST_JSON_PATH.resolve()
    != C_TEST_TARGET.resolve()
):
    shutil.copy2(
        TEST_JSON_PATH,
        C_TEST_TARGET,
    )

TEST_JSON_PATH = C_TEST_TARGET

# -------------------------------------------------------------------------
# Load official test JSON.
# -------------------------------------------------------------------------
with open(
    TEST_JSON_PATH,
    "r",
    encoding="utf-8",
) as f:
    TEST_JSON = json.load(f)

# -------------------------------------------------------------------------
# Normalize supported JSON structures.
# -------------------------------------------------------------------------
if isinstance(
    TEST_JSON,
    list,
):
    OFFICIAL_TEST_RECORDS = (
        TEST_JSON
    )

elif isinstance(
    TEST_JSON,
    dict,
):
    OFFICIAL_TEST_RECORDS = None

    for _key in (
        "data",
        "records",
        "test",
        "examples",
    ):
        if isinstance(
            TEST_JSON.get(_key),
            list,
        ):
            OFFICIAL_TEST_RECORDS = (
                TEST_JSON[_key]
            )
            break

    if OFFICIAL_TEST_RECORDS is None:
        OFFICIAL_TEST_RECORDS = []

        for _test_id, _value in (
            TEST_JSON.items()
        ):
            if isinstance(
                _value,
                dict,
            ):
                _record = dict(
                    _value
                )

                _record.setdefault(
                    "id",
                    str(_test_id),
                )

            else:
                _record = {
                    "id": str(_test_id),
                    "prompt": str(_value),
                }

            OFFICIAL_TEST_RECORDS.append(
                _record
            )

else:
    raise TypeError(
        "Unsupported official test JSON type: "
        f"{type(TEST_JSON)}"
    )

assert OFFICIAL_TEST_RECORDS, (
    "Official test JSON contains no records."
)

# -------------------------------------------------------------------------
# Resolve ID and prompt.
# -------------------------------------------------------------------------
def get_test_id(
    record
):
    for _key in (
        "id",
        "ID",
        "test_id",
        "testId",
    ):
        if _key in record:
            return str(
                record[_key]
            )

    raise KeyError(
        "No test ID field found. "
        f"Keys: {list(record.keys())}"
    )


def get_test_prompt(
    record
):
    for _key in (
        "prompt",
        "text",
        "input",
    ):
        if _key in record:
            return str(
                record[_key]
            )

    raise KeyError(
        "No test prompt field found. "
        f"Keys: {list(record.keys())}"
    )

# -------------------------------------------------------------------------
# Canonical test arrays.
# -------------------------------------------------------------------------
OFFICIAL_TEST_IDS = [
    get_test_id(record)
    for record in OFFICIAL_TEST_RECORDS
]

OFFICIAL_TEST_PROMPTS = [
    get_test_prompt(record)
    for record in OFFICIAL_TEST_RECORDS
]

assert len(
    OFFICIAL_TEST_IDS
) == len(
    set(OFFICIAL_TEST_IDS)
), (
    "Duplicate official test IDs detected."
)

assert all(
    prompt.strip()
    for prompt in OFFICIAL_TEST_PROMPTS
), (
    "At least one official test prompt is empty."
)

# -------------------------------------------------------------------------
# Verify 512-word official prompts.
# -------------------------------------------------------------------------
OFFICIAL_PROMPT_WORD_COUNTS = [
    len(
        prompt.strip().split()
    )
    for prompt in OFFICIAL_TEST_PROMPTS
]

assert all(
    count == 512
    for count in OFFICIAL_PROMPT_WORD_COUNTS
), (
    "At least one official test prompt does not contain exactly "
    "512 whitespace-separated words."
)

# -------------------------------------------------------------------------
# Select active records according to run mode.
# -------------------------------------------------------------------------
if RUN_MODE == "fast":
    ACTIVE_TEST_RECORDS = (
        OFFICIAL_TEST_RECORDS[:2]
    )
else:
    ACTIVE_TEST_RECORDS = (
        OFFICIAL_TEST_RECORDS
    )

EXPECTED_TEST_COUNT = (
    2
    if RUN_MODE == "fast"
    else 1000
)

assert len(
    ACTIVE_TEST_RECORDS
) == EXPECTED_TEST_COUNT, (
    f"Expected {EXPECTED_TEST_COUNT} active records "
    f"for RUN_MODE={RUN_MODE}, found "
    f"{len(ACTIVE_TEST_RECORDS)}."
)

ACTIVE_TEST_IDS = [
    get_test_id(record)
    for record in ACTIVE_TEST_RECORDS
]

ACTIVE_TEST_PROMPTS = [
    get_test_prompt(record)
    for record in ACTIVE_TEST_RECORDS
]

# -------------------------------------------------------------------------
# Save test manifest.
# -------------------------------------------------------------------------
TEST_MANIFEST = {
    "experiment_id": EXPERIMENT_ID,
    "architecture_name": ARCHITECTURE_NAME,
    "huggingface_repo": HF_REPO_ID,
    "test_json_path": str(
        TEST_JSON_PATH
    ),
    "total_official_examples": int(
        len(OFFICIAL_TEST_RECORDS)
    ),
    "active_examples": int(
        len(ACTIVE_TEST_RECORDS)
    ),
    "prompt_word_count": 512,
    "active_test_ids": list(
        ACTIVE_TEST_IDS
    ),
}

TEST_MANIFEST_PATH = (
    CONFIGS_DIR /
    "official_test_manifest_C.json"
)

with open(
    TEST_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        TEST_MANIFEST,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(
    "\n✅ Official Phase-1 test set loaded."
)

print(
    f"Official examples : "
    f"{len(OFFICIAL_TEST_RECORDS):,}"
)

print(
    f"Active examples   : "
    f"{len(ACTIVE_TEST_RECORDS):,}"
)

print(
    f"Prompt words      : "
    f"{min(OFFICIAL_PROMPT_WORD_COUNTS)}–"
    f"{max(OFFICIAL_PROMPT_WORD_COUNTS)}"
)

print(
    f"Target generation : "
    f"{GENERATION_CONFIG['target_words']} words"
)

print(
    f"Test manifest     : "
    f"{TEST_MANIFEST_PATH}"
)

📄 Official test file: /kaggle/input/datasets/rohitbatra2004/phase133/architecture_C/test_data/Phase_1_Pretraining/Test/test.json

✅ Official Phase-1 test set loaded.
Official examples : 1,000
Active examples   : 1,000
Prompt words      : 512–512
Target generation : 128 words
Test manifest     : /kaggle/working/architecture_C/configs/official_test_manifest_C.json


In [16]:
# =========================================================================
# Cell 14.5 — Restore Architecture-C generation checkpoint from Kaggle Dataset
# DECISION: ADD
#
# Purpose:
#   Restore prediction_C_partial.json and generation_progress_C.json
#   from the persisted Kaggle Dataset into /kaggle/working.
#
# IMPORTANT:
#   - Does NOT retrain the model.
#   - Does NOT modify model weights.
#   - Does NOT overwrite a working checkpoint with an unknown file.
#   - If multiple candidate files are found, it stops instead of guessing.
# =========================================================================

from pathlib import Path
import json
import shutil


print(
    "♻️ Checking for saved Architecture-C generation checkpoint..."
)


# =========================================================================
# WORKING PATHS
# =========================================================================

WORKING_ROOT = Path(
    "/kaggle/working/architecture_C"
)

WORKING_PREDICTIONS_DIR = (
    WORKING_ROOT / "predictions"
)

WORKING_METRICS_DIR = (
    WORKING_ROOT / "metrics"
)

WORKING_PREDICTIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

WORKING_METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


WORKING_PARTIAL_PATH = (
    WORKING_PREDICTIONS_DIR
    / "prediction_C_partial.json"
)

WORKING_PROGRESS_PATH = (
    WORKING_METRICS_DIR
    / "generation_progress_C.json"
)


# =========================================================================
# FIND PERSISTED PARTIAL PREDICTION FILE
# =========================================================================

partial_candidates = sorted(
    Path("/kaggle/input").rglob(
        "prediction_C_partial.json"
    )
)

print(
    f"Partial prediction candidates : "
    f"{len(partial_candidates)}"
)

if len(partial_candidates) > 1:

    print()
    print(
        "❌ Multiple prediction_C_partial.json files found:"
    )

    for path in partial_candidates:
        print(
            f"   {path}"
        )

    raise RuntimeError(
        "Multiple Architecture-C partial prediction files "
        "were found. Stopping instead of guessing which one "
        "should be restored."
    )


# =========================================================================
# FIND PERSISTED PROGRESS FILE
# =========================================================================

progress_candidates = sorted(
    Path("/kaggle/input").rglob(
        "generation_progress_C.json"
    )
)

print(
    f"Progress file candidates       : "
    f"{len(progress_candidates)}"
)

if len(progress_candidates) > 1:

    print()
    print(
        "❌ Multiple generation_progress_C.json files found:"
    )

    for path in progress_candidates:
        print(
            f"   {path}"
        )

    raise RuntimeError(
        "Multiple Architecture-C progress files "
        "were found. Stopping instead of guessing."
    )


# =========================================================================
# RESTORE PARTIAL PREDICTIONS
# =========================================================================

if len(partial_candidates) == 1:

    source_partial = (
        partial_candidates[0]
    )

    shutil.copy2(
        source_partial,
        WORKING_PARTIAL_PATH,
    )

    print()
    print(
        "✅ Partial predictions restored."
    )

    print(
        f"   From : {source_partial}"
    )

    print(
        f"   To   : {WORKING_PARTIAL_PATH}"
    )

else:

    print()
    print(
        "ℹ️ No persisted prediction_C_partial.json "
        "was found."
    )


# =========================================================================
# RESTORE OR REBUILD PROGRESS FILE
# =========================================================================

if len(progress_candidates) == 1:

    source_progress = (
        progress_candidates[0]
    )

    shutil.copy2(
        source_progress,
        WORKING_PROGRESS_PATH,
    )

    print()
    print(
        "✅ Generation progress restored."
    )

    print(
        f"   From : {source_progress}"
    )

    print(
        f"   To   : {WORKING_PROGRESS_PATH}"
    )


elif WORKING_PARTIAL_PATH.is_file():

    # ---------------------------------------------------------------------
    # If the dataset contains the partial prediction file but the progress
    # file was not persisted, reconstruct the minimal progress metadata
    # required by the existing Architecture-C generation cell.
    # ---------------------------------------------------------------------

    with open(
        WORKING_PARTIAL_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        _restored_predictions = json.load(
            f
        )

    assert isinstance(
        _restored_predictions,
        dict,
    ), (
        "prediction_C_partial.json must contain "
        "a dictionary keyed by test ID."
    )


    _expected_ids = {
        str(
            _test_id(record)
        )
        for record in ACTIVE_TEST_RECORDS
    }


    # ---------------------------------------------------------------------
    # Validate IDs.
    # ---------------------------------------------------------------------
    _actual_ids = {
        str(test_id)
        for test_id in _restored_predictions.keys()
    }

    assert (
        _actual_ids
        <= _expected_ids
    ), (
        "Persisted Architecture-C partial prediction "
        "contains unknown test IDs.\n"
        f"Unknown IDs: "
        f"{sorted(_actual_ids - _expected_ids)[:20]}"
    )


    # ---------------------------------------------------------------------
    # Validate every already-saved prediction.
    # ---------------------------------------------------------------------
    for _test_id_value, _prediction in (
        _restored_predictions.items()
    ):

        assert isinstance(
            _prediction,
            dict,
        ), (
            f"Prediction {_test_id_value} "
            "is not a dictionary."
        )

        assert set(
            _prediction.keys()
        ) == {
            "text"
        }, (
            f"Prediction {_test_id_value} "
            "does not contain exactly {'text'}."
        )

        assert isinstance(
            _prediction["text"],
            str,
        ), (
            f"Prediction {_test_id_value} "
            "text is not a string."
        )

        _word_count = count_words(
            _prediction["text"]
        )

        assert (
            _word_count
            == int(
                GENERATION_CONFIG[
                    "target_words"
                ]
            )
        ), (
            f"Prediction {_test_id_value} has "
            f"{_word_count} words instead of "
            f"{GENERATION_CONFIG['target_words']}."
        )


    # ---------------------------------------------------------------------
    # Reconstruct progress metadata compatible with Cell 15.
    # ---------------------------------------------------------------------
    generation_progress = {
        "experiment_id": EXPERIMENT_ID,
        "architecture_name": ARCHITECTURE_NAME,
        "architecture_version": ARCHITECTURE_VERSION,
        "run_mode": RUN_MODE,
        "target_words": int(
            GENERATION_CONFIG[
                "target_words"
            ]
        ),
        "max_new_tokens": int(
            GENERATION_CONFIG[
                "max_new_tokens"
            ]
        ),
        "temperature": float(
            GENERATION_CONFIG[
                "temperature"
            ]
        ),
        "top_p": float(
            GENERATION_CONFIG[
                "top_p"
            ]
        ),
        "top_k": int(
            GENERATION_CONFIG[
                "top_k"
            ]
        ),
        "repetition_penalty": float(
            GENERATION_CONFIG[
                "repetition_penalty"
            ]
        ),
        "completed_ids": sorted(
            _actual_ids
        ),
        "completed_count": len(
            _actual_ids
        ),
        "status": "resuming",
    }


    with open(
        WORKING_PROGRESS_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            generation_progress,
            f,
            ensure_ascii=False,
            indent=2,
        )


    print()
    print(
        "✅ Progress file rebuilt from the partial "
        "prediction file."
    )


elif not WORKING_PROGRESS_PATH.is_file():

    print()
    print(
        "ℹ️ No saved Architecture-C generation "
        "checkpoint exists yet."
    )


# =========================================================================
# FINAL RESUME CHECK
# =========================================================================

print()
print(
    "============================================================"
)

print(
    "Architecture-C resume status"
)

print(
    "============================================================"
)

print(
    f"Partial file : "
    f"{'✅' if WORKING_PARTIAL_PATH.is_file() else '❌'}"
)

print(
    f"Progress file: "
    f"{'✅' if WORKING_PROGRESS_PATH.is_file() else '❌'}"
)


# =========================================================================
# REPORT NUMBER OF COMPLETED EXAMPLES
# =========================================================================

if WORKING_PARTIAL_PATH.is_file():

    with open(
        WORKING_PARTIAL_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        _check_predictions = json.load(
            f
        )

    print()
    print(
        f"Already completed : "
        f"{len(_check_predictions):,}/"
        f"{len(ACTIVE_TEST_RECORDS):,}"
    )

    if _check_predictions:

        _saved_ids = [
            str(x)
            for x in _check_predictions.keys()
        ]

        print(
            f"First saved ID    : "
            f"{_saved_ids[0]}"
        )

        print(
            f"Last saved ID     : "
            f"{_saved_ids[-1]}"
        )

else:

    print()
    print(
        "Already completed : 0"
    )


print(
    "============================================================"
)

print(
    "✅ Architecture-C checkpoint restore step finished."
)

print(
    "Run the Architecture-C generation cell next."
)

♻️ Checking for saved Architecture-C generation checkpoint...
Partial prediction candidates : 1
Progress file candidates       : 1

✅ Partial predictions restored.
   From : /kaggle/input/datasets/rohitbatra2004/phase133/architecture_C/predictions/prediction_C_partial.json
   To   : /kaggle/working/architecture_C/predictions/prediction_C_partial.json

✅ Generation progress restored.
   From : /kaggle/input/datasets/rohitbatra2004/phase133/architecture_C/metrics/generation_progress_C.json
   To   : /kaggle/working/architecture_C/metrics/generation_progress_C.json

Architecture-C resume status
Partial file : ✅
Progress file: ✅

Already completed : 118/1,000
First saved ID    : 1
Last saved ID     : 118
✅ Architecture-C checkpoint restore step finished.
Run the Architecture-C generation cell next.


In [17]:
# Cell 15 — Generate Architecture-C official-test continuations with automatic resume
# DECISION: ADD

import hashlib
import json
import random
from pathlib import Path

print("🚀 Starting Architecture-C official-test generation...")

# -------------------------------------------------------------------------
# Reconstruct output directories explicitly.
# -------------------------------------------------------------------------
NOTEBOOK_DIR = Path(
    globals().get(
        "NOTEBOOK_DIR",
        "/kaggle/working/architecture_C",
    )
)

METRICS_DIR = NOTEBOOK_DIR / "metrics"
PREDICTIONS_DIR = NOTEBOOK_DIR / "predictions"

METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PREDICTIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# -------------------------------------------------------------------------
# Ensure the best checkpoint is loaded for inference.
# -------------------------------------------------------------------------
assert BEST_CHECKPOINT_PATH.is_file(), (
    f"Best checkpoint not found: "
    f"{BEST_CHECKPOINT_PATH}"
)

BEST_CHECKPOINT = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

assert checkpoint_is_compatible(
    BEST_CHECKPOINT
), (
    "Best Architecture-C checkpoint is incompatible "
    "with the current configuration."
)

MODEL.load_state_dict(
    BEST_CHECKPOINT["model"]
)

MODEL = MODEL.to(
    DEVICE
)

MODEL.eval()

# -------------------------------------------------------------------------
# Ensure the tokenizer exists in the current runtime.
# -------------------------------------------------------------------------
if "tokenizer" not in globals():
    from tokenizers import Tokenizer

    tokenizer = Tokenizer.from_file(
        str(TOKENIZER_PATH)
    )

# -------------------------------------------------------------------------
# Ensure the test records exist.
# -------------------------------------------------------------------------
assert "ACTIVE_TEST_RECORDS" in globals(), (
    "ACTIVE_TEST_RECORDS is unavailable. "
    "Run Cell 14 first."
)

assert len(
    ACTIVE_TEST_RECORDS
) > 0

# -------------------------------------------------------------------------
# Generation function.
# -------------------------------------------------------------------------
GENERATE_C = (
    generate_architecture_C_continuation
)

assert callable(
    GENERATE_C
), (
    "Architecture-C generation helper is unavailable. "
    "Run Cell 13 first."
)

# -------------------------------------------------------------------------
# Artifact paths.
# -------------------------------------------------------------------------
GENERATION_PROGRESS_PATH = (
    METRICS_DIR /
    "generation_progress_C.json"
)

PARTIAL_PREDICTION_PATH = (
    PREDICTIONS_DIR /
    "prediction_C_partial.json"
)

FINAL_PREDICTION_PATH = (
    PREDICTIONS_DIR /
    "prediction_C.json"
)

# -------------------------------------------------------------------------
# Resolve official test fields.
# -------------------------------------------------------------------------
def _test_id(record):
    for key in (
        "id",
        "ID",
        "test_id",
        "testId",
    ):
        if key in record:
            return str(
                record[key]
            )

    raise KeyError(
        f"Test ID not found. Keys: {list(record.keys())}"
    )


def _test_prompt(record):
    for key in (
        "prompt",
        "text",
        "input",
    ):
        if key in record:
            return str(
                record[key]
            )

    raise KeyError(
        f"Test prompt not found. Keys: {list(record.keys())}"
    )


ACTIVE_TEST_IDS = [
    _test_id(record)
    for record in ACTIVE_TEST_RECORDS
]

assert len(
    ACTIVE_TEST_IDS
) == len(
    set(ACTIVE_TEST_IDS)
), "Duplicate active test IDs detected."

# -------------------------------------------------------------------------
# Stable per-example seed.
# -------------------------------------------------------------------------
def _stable_generation_seed(
    test_id
):
    digest = hashlib.sha256(
        (
            f"{EXPERIMENT_ID}|"
            f"{ARCHITECTURE_NAME}|"
            f"{test_id}"
        ).encode("utf-8")
    ).digest()

    return (
        int.from_bytes(
            digest[:8],
            byteorder="big",
            signed=False,
        )
        % (2**31 - 1)
    )


# -------------------------------------------------------------------------
# Automatic resume.
# -------------------------------------------------------------------------
predictions = {}

if (
    GENERATION_PROGRESS_PATH.is_file()
    and PARTIAL_PREDICTION_PATH.is_file()
):
    with open(
        GENERATION_PROGRESS_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        generation_progress = json.load(f)

    with open(
        PARTIAL_PREDICTION_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        predictions = json.load(f)

    assert (
        generation_progress["experiment_id"]
        == EXPERIMENT_ID
    ), (
        "Existing generation checkpoint belongs "
        "to another experiment."
    )

    assert (
        generation_progress["architecture_name"]
        == ARCHITECTURE_NAME
    ), (
        "Existing generation checkpoint belongs "
        "to another architecture."
    )

    assert (
        generation_progress["run_mode"]
        == RUN_MODE
    ), (
        "Existing generation checkpoint has "
        "a different run mode."
    )

    assert (
        int(
            generation_progress[
                "target_words"
            ]
        )
        == int(
            GENERATION_CONFIG[
                "target_words"
            ]
        )
    ), (
        "Existing generation checkpoint uses "
        "a different target word count."
    )

    print(
        f"♻️ Resuming generation: "
        f"{len(predictions):,}/"
        f"{len(ACTIVE_TEST_RECORDS):,} complete."
    )

else:
    generation_progress = {
        "experiment_id": EXPERIMENT_ID,
        "architecture_name": ARCHITECTURE_NAME,
        "architecture_version": ARCHITECTURE_VERSION,
        "run_mode": RUN_MODE,
        "target_words": int(
            GENERATION_CONFIG[
                "target_words"
            ]
        ),
        "max_new_tokens": int(
            GENERATION_CONFIG[
                "max_new_tokens"
            ]
        ),
        "temperature": float(
            GENERATION_CONFIG[
                "temperature"
            ]
        ),
        "top_p": float(
            GENERATION_CONFIG[
                "top_p"
            ]
        ),
        "top_k": int(
            GENERATION_CONFIG[
                "top_k"
            ]
        ),
        "repetition_penalty": float(
            GENERATION_CONFIG[
                "repetition_penalty"
            ]
        ),
        "completed_ids": [],
        "completed_count": 0,
        "status": "running",
    }

# -------------------------------------------------------------------------
# Generate remaining official examples.
# -------------------------------------------------------------------------
completed_ids = set(
    predictions.keys()
)

for index, record in enumerate(
    ACTIVE_TEST_RECORDS,
    start=1,
):
    test_id = _test_id(record)
    prompt = _test_prompt(record)

    if test_id in completed_ids:
        continue

    # Deterministic example-local RNG.
    seed = _stable_generation_seed(
        test_id
    )

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # -------------------------------------------------------------
    # Generate exactly the configured word count.
    # -------------------------------------------------------------
    continuation = GENERATE_C(
        prompt
    )

    continuation = str(
        continuation
    ).strip()

    assert continuation, (
        f"Empty continuation generated "
        f"for test ID {test_id}."
    )

    generated_word_count = count_words(
        continuation
    )

    assert (
        generated_word_count
        == GENERATION_CONFIG[
            "target_words"
        ]
    ), (
        f"Test ID {test_id}: expected "
        f"{GENERATION_CONFIG['target_words']} words, "
        f"got {generated_word_count}."
    )

    predictions[test_id] = {
        "text": continuation
    }

    completed_ids.add(
        test_id
    )

    # -------------------------------------------------------------
    # Persist immediately after each successful example.
    # -------------------------------------------------------------
    generation_progress[
        "completed_ids"
    ] = sorted(
        completed_ids
    )

    generation_progress[
        "completed_count"
    ] = len(
        completed_ids
    )

    generation_progress[
        "last_completed_id"
    ] = test_id

    with open(
        PARTIAL_PREDICTION_PATH,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            predictions,
            f,
            ensure_ascii=False,
            indent=2,
        )

    with open(
        GENERATION_PROGRESS_PATH,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            generation_progress,
            f,
            ensure_ascii=False,
            indent=2,
        )

    print(
        f"[{index:>4}/"
        f"{len(ACTIVE_TEST_RECORDS):<4}] "
        f"ID={test_id} | "
        f"{generated_word_count} words"
    )

# -------------------------------------------------------------------------
# Final validation.
# -------------------------------------------------------------------------
EXPECTED_IDS = set(
    ACTIVE_TEST_IDS
)

assert set(
    predictions.keys()
) == EXPECTED_IDS, (
    "Prediction IDs do not exactly match "
    "the active official test IDs."
)

for test_id, prediction in (
    predictions.items()
):
    assert isinstance(
        prediction,
        dict,
    )

    assert set(
        prediction.keys()
    ) == {
        "text"
    }

    assert isinstance(
        prediction["text"],
        str,
    )

    assert (
        count_words(
            prediction["text"]
        )
        == GENERATION_CONFIG[
            "target_words"
        ]
    ), (
        f"Prediction {test_id} has an "
        f"incorrect word count."
    )

# -------------------------------------------------------------------------
# Save ordered final prediction file.
# -------------------------------------------------------------------------
ordered_predictions = {
    _test_id(record): predictions[
        _test_id(record)
    ]
    for record in ACTIVE_TEST_RECORDS
}

with open(
    FINAL_PREDICTION_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        ordered_predictions,
        f,
        ensure_ascii=False,
        indent=2,
    )

generation_progress[
    "completed_ids"
] = sorted(
    completed_ids
)

generation_progress[
    "completed_count"
] = len(
    completed_ids
)

generation_progress[
    "status"
] = "complete"

with open(
    GENERATION_PROGRESS_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        generation_progress,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("\n✅ Architecture-C generation complete.")
print(
    f"Examples generated : "
    f"{len(predictions):,}"
)
print(
    f"Words/example      : "
    f"{GENERATION_CONFIG['target_words']}"
)
print(
    f"Prediction file    : "
    f"{FINAL_PREDICTION_PATH}"
)

🚀 Starting Architecture-C official-test generation...
♻️ Resuming generation: 118/1,000 complete.
[ 119/1000] ID=119 | 128 words
[ 120/1000] ID=120 | 128 words
[ 121/1000] ID=121 | 128 words
[ 122/1000] ID=122 | 128 words
[ 123/1000] ID=123 | 128 words
[ 124/1000] ID=124 | 128 words
[ 125/1000] ID=125 | 128 words
[ 126/1000] ID=126 | 128 words
[ 127/1000] ID=127 | 128 words
[ 128/1000] ID=128 | 128 words
[ 129/1000] ID=129 | 128 words
[ 130/1000] ID=130 | 128 words
[ 131/1000] ID=131 | 128 words
[ 132/1000] ID=132 | 128 words
[ 133/1000] ID=133 | 128 words
[ 134/1000] ID=134 | 128 words
[ 135/1000] ID=135 | 128 words
[ 136/1000] ID=136 | 128 words
[ 137/1000] ID=137 | 128 words
[ 138/1000] ID=138 | 128 words
[ 139/1000] ID=139 | 128 words
[ 140/1000] ID=140 | 128 words
[ 141/1000] ID=141 | 128 words
[ 142/1000] ID=142 | 128 words
[ 143/1000] ID=143 | 128 words
[ 144/1000] ID=144 | 128 words
[ 145/1000] ID=145 | 128 words
[ 146/1000] ID=146 | 128 words
[ 147/1000] ID=147 | 128 words
[ 1

In [18]:
# Cell 16 — Validate Architecture-C predictions and create submission.zip
# DECISION: UPDATE

import json
import zipfile
from pathlib import Path

print("📦 Packaging Architecture-C submission...")

# -------------------------------------------------------------------------
# Explicit artifact paths.
# -------------------------------------------------------------------------
NOTEBOOK_DIR = Path(
    "/kaggle/working/architecture_C"
)

PREDICTIONS_DIR = (
    NOTEBOOK_DIR / "predictions"
)

SUBMISSION_DIR = (
    NOTEBOOK_DIR / "submission"
)

METRICS_DIR = (
    NOTEBOOK_DIR / "metrics"
)

SUBMISSION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FINAL_PREDICTION_PATH = (
    PREDICTIONS_DIR /
    "prediction_C.json"
)

SUBMISSION_PREDICTION_PATH = (
    SUBMISSION_DIR /
    "prediction.json"
)

SUBMISSION_ZIP_PATH = (
    SUBMISSION_DIR /
    "submission.zip"
)

# -------------------------------------------------------------------------
# Require completed predictions.
# -------------------------------------------------------------------------
assert FINAL_PREDICTION_PATH.is_file(), (
    f"Architecture-C prediction file not found: "
    f"{FINAL_PREDICTION_PATH}"
)

with open(
    FINAL_PREDICTION_PATH,
    "r",
    encoding="utf-8",
) as f:
    predictions = json.load(f)

assert isinstance(
    predictions,
    dict,
), (
    "Architecture-C prediction file must contain a JSON object."
)

# -------------------------------------------------------------------------
# Expected official IDs for the current run.
# -------------------------------------------------------------------------
assert "ACTIVE_TEST_IDS" in globals(), (
    "ACTIVE_TEST_IDS is unavailable. Run Cell 14 first."
)

EXPECTED_IDS = set(
    ACTIVE_TEST_IDS
)

EXPECTED_COUNT = len(
    ACTIVE_TEST_IDS
)

if RUN_MODE == "production":
    assert EXPECTED_COUNT == 1000, (
        f"Production must contain 1,000 official test IDs, "
        f"found {EXPECTED_COUNT}."
    )

assert len(predictions) == (
    EXPECTED_COUNT
), (
    f"Expected {EXPECTED_COUNT} predictions, "
    f"found {len(predictions)}."
)

assert set(
    predictions.keys()
) == EXPECTED_IDS, (
    "Prediction IDs do not exactly match "
    "the active official test IDs."
)

# -------------------------------------------------------------------------
# Validate every prediction against the required schema.
# -------------------------------------------------------------------------
TARGET_WORDS = int(
    GENERATION_CONFIG["target_words"]
)

for _test_id in ACTIVE_TEST_IDS:
    _prediction = predictions[
        _test_id
    ]

    assert isinstance(
        _prediction,
        dict,
    ), (
        f"Prediction {_test_id} must be a JSON object."
    )

    assert set(
        _prediction.keys()
    ) == {
        "text"
    }, (
        f"Prediction {_test_id} must contain exactly "
        "the key 'text'."
    )

    _text = _prediction[
        "text"
    ]

    assert isinstance(
        _text,
        str,
    ), (
        f"Prediction {_test_id} text must be a string."
    )

    _text = _text.strip()

    assert _text, (
        f"Prediction {_test_id} is empty."
    )

    assert count_words(
        _text
    ) == TARGET_WORDS, (
        f"Prediction {_test_id} contains "
        f"{count_words(_text)} words; "
        f"expected {TARGET_WORDS}."
    )

# -------------------------------------------------------------------------
# Write the exact prediction.json required for submission.
# -------------------------------------------------------------------------
with open(
    SUBMISSION_PREDICTION_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        {
            _test_id: {
                "text": predictions[
                    _test_id
                ]["text"].strip()
            }
            for _test_id in ACTIVE_TEST_IDS
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

# -------------------------------------------------------------------------
# Recreate submission.zip.
# prediction.json is placed directly at the ZIP root.
# -------------------------------------------------------------------------
TEMP_ZIP_PATH = (
    SUBMISSION_DIR /
    "submission_tmp.zip"
)

if TEMP_ZIP_PATH.exists():
    TEMP_ZIP_PATH.unlink()

if SUBMISSION_ZIP_PATH.exists():
    SUBMISSION_ZIP_PATH.unlink()

with zipfile.ZipFile(
    TEMP_ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as _zip:
    _zip.write(
        SUBMISSION_PREDICTION_PATH,
        arcname="prediction.json",
    )

TEMP_ZIP_PATH.replace(
    SUBMISSION_ZIP_PATH
)

# -------------------------------------------------------------------------
# Verify ZIP structure and contents.
# -------------------------------------------------------------------------
with zipfile.ZipFile(
    SUBMISSION_ZIP_PATH,
    mode="r",
) as _zip:

    ZIP_MEMBERS = _zip.namelist()

    assert ZIP_MEMBERS == [
        "prediction.json"
    ], (
        "submission.zip must contain "
        "prediction.json directly at its root."
    )

    with _zip.open(
        "prediction.json",
        "r",
    ) as _f:
        ZIPPED_PREDICTIONS = json.load(
            _f
        )

# Compare the exact serialized prediction structure.
with open(
    SUBMISSION_PREDICTION_PATH,
    "r",
    encoding="utf-8",
) as f:
    WRITTEN_PREDICTIONS = json.load(
        f
    )

assert ZIPPED_PREDICTIONS == (
    WRITTEN_PREDICTIONS
), (
    "prediction.json inside submission.zip "
    "does not match the written prediction.json."
)

# -------------------------------------------------------------------------
# Final submission metadata.
# -------------------------------------------------------------------------
PACKAGING_METADATA = {
    "experiment_id": EXPERIMENT_ID,
    "architecture_name": ARCHITECTURE_NAME,
    "architecture_version": ARCHITECTURE_VERSION,
    "run_mode": RUN_MODE,
    "prediction_count": len(
        predictions
    ),
    "target_words": TARGET_WORDS,
    "prediction_json": str(
        SUBMISSION_PREDICTION_PATH
    ),
    "submission_zip": str(
        SUBMISSION_ZIP_PATH
    ),
    "zip_members": ZIP_MEMBERS,
    "official_test_ids_validated": True,
    "exact_word_count_validated": True,
}

PACKAGING_METADATA_PATH = (
    METRICS_DIR /
    "submission_packaging_C.json"
)

with open(
    PACKAGING_METADATA_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        PACKAGING_METADATA,
        f,
        ensure_ascii=False,
        indent=2,
    )

# -------------------------------------------------------------------------
# Final output.
# -------------------------------------------------------------------------
print(
    "\n✅ Architecture-C submission package created."
)

print(
    f"Predictions     : "
    f"{len(predictions):,}"
)

print(
    f"Words/example   : "
    f"{TARGET_WORDS}"
)

print(
    f"prediction.json : "
    f"{SUBMISSION_PREDICTION_PATH}"
)

print(
    f"submission.zip  : "
    f"{SUBMISSION_ZIP_PATH}"
)

print(
    f"ZIP contents    : "
    f"{ZIP_MEMBERS}"
)

📦 Packaging Architecture-C submission...

✅ Architecture-C submission package created.
Predictions     : 1,000
Words/example   : 128
prediction.json : /kaggle/working/architecture_C/submission/prediction.json
submission.zip  : /kaggle/working/architecture_C/submission/submission.zip
ZIP contents    : ['prediction.json']


In [19]:
# Cell 17 — Finalize Architecture-C experiment manifest and artifact inventory
# DECISION: UPDATE

import json
from pathlib import Path

print("🧾 Finalizing Architecture-C experiment manifest...")

# -------------------------------------------------------------------------
# Explicit Architecture-C directories.
# -------------------------------------------------------------------------
NOTEBOOK_DIR = Path(
    "/kaggle/working/architecture_C"
)

CHECKPOINTS_DIR = (
    NOTEBOOK_DIR / "checkpoints"
)

METRICS_DIR = (
    NOTEBOOK_DIR / "metrics"
)

PREDICTIONS_DIR = (
    NOTEBOOK_DIR / "predictions"
)

SUBMISSION_DIR = (
    NOTEBOOK_DIR / "submission"
)

# -------------------------------------------------------------------------
# Required artifacts.
# -------------------------------------------------------------------------
REQUIRED_ARTIFACTS = {
    "final_model": (
        NOTEBOOK_DIR /
        "model_C_final.pt"
    ),
    "final_state_dict": (
        NOTEBOOK_DIR /
        "model_C_state_dict.pt"
    ),
    "model_metadata": (
        NOTEBOOK_DIR /
        "model_C_metadata.json"
    ),
    "training_summary": (
        METRICS_DIR /
        "training_summary_C.json"
    ),
    "selected_checkpoint": (
        METRICS_DIR /
        "selected_checkpoint_C.json"
    ),
    "prediction_json": (
        PREDICTIONS_DIR /
        "prediction_C.json"
    ),
    "submission_prediction": (
        SUBMISSION_DIR /
        "prediction.json"
    ),
    "submission_zip": (
        SUBMISSION_DIR /
        "submission.zip"
    ),
    "submission_metadata": (
        METRICS_DIR /
        "submission_packaging_C.json"
    ),
}

# -------------------------------------------------------------------------
# Verify all required artifacts.
# -------------------------------------------------------------------------
for _name, _path in REQUIRED_ARTIFACTS.items():
    assert _path.is_file(), (
        f"Required Architecture-C artifact missing: "
        f"{_name} -> {_path}"
    )

# -------------------------------------------------------------------------
# Load recorded metadata.
# -------------------------------------------------------------------------
with open(
    REQUIRED_ARTIFACTS["model_metadata"],
    "r",
    encoding="utf-8",
) as f:
    MODEL_METADATA = json.load(f)

with open(
    REQUIRED_ARTIFACTS["training_summary"],
    "r",
    encoding="utf-8",
) as f:
    TRAINING_SUMMARY = json.load(f)

with open(
    REQUIRED_ARTIFACTS["selected_checkpoint"],
    "r",
    encoding="utf-8",
) as f:
    CHECKPOINT_METADATA = json.load(f)

with open(
    REQUIRED_ARTIFACTS["submission_metadata"],
    "r",
    encoding="utf-8",
) as f:
    SUBMISSION_METADATA = json.load(f)

# -------------------------------------------------------------------------
# Validate experiment identity.
# -------------------------------------------------------------------------
assert (
    MODEL_METADATA["experiment_id"]
    == EXPERIMENT_ID
)

assert (
    TRAINING_SUMMARY["experiment_id"]
    == EXPERIMENT_ID
)

assert (
    CHECKPOINT_METADATA["experiment_id"]
    == EXPERIMENT_ID
)

assert (
    SUBMISSION_METADATA["experiment_id"]
    == EXPERIMENT_ID
)

assert (
    MODEL_METADATA["architecture_name"]
    == ARCHITECTURE_NAME
)

assert (
    TRAINING_SUMMARY["architecture_name"]
    == ARCHITECTURE_NAME
)

# -------------------------------------------------------------------------
# Production-specific requirement checks.
# -------------------------------------------------------------------------
if RUN_MODE == "production":

    assert 50_000_000 <= (
        int(
            MODEL_METADATA[
                "parameter_count"
            ]
        )
    ) <= 300_000_000, (
        "Production Architecture-C model is outside "
        "the required 50M–300M parameter range."
    )

    assert int(
        SUBMISSION_METADATA[
            "prediction_count"
        ]
    ) == 1000, (
        "Production Architecture-C must contain "
        "exactly 1,000 official predictions."
    )

    assert int(
        SUBMISSION_METADATA[
            "target_words"
        ]
    ) == 128, (
        "Production Architecture-C predictions must "
        "contain exactly 128 words each."
    )

# -------------------------------------------------------------------------
# Checkpoint inventory.
# -------------------------------------------------------------------------
CHECKPOINT_FILES = sorted(
    str(
        _path.relative_to(
            NOTEBOOK_DIR
        )
    )
    for _path in CHECKPOINTS_DIR.glob(
        "*.pt"
    )
    if _path.is_file()
)

assert CHECKPOINT_FILES, (
    "No Architecture-C checkpoint files were found."
)

# -------------------------------------------------------------------------
# Artifact inventory with sizes.
# -------------------------------------------------------------------------
ARTIFACT_INVENTORY = {}

for _name, _path in REQUIRED_ARTIFACTS.items():
    ARTIFACT_INVENTORY[_name] = {
        "path": str(
            _path.relative_to(
                NOTEBOOK_DIR
            )
        ),
        "size_bytes": int(
            _path.stat().st_size
        ),
    }

# -------------------------------------------------------------------------
# Final experiment manifest.
# -------------------------------------------------------------------------
FINAL_EXPERIMENT_MANIFEST = {
    "project": "CS781 Phase 1",
    "experiment_id": EXPERIMENT_ID,
    "architecture_name": ARCHITECTURE_NAME,
    "architecture_version": ARCHITECTURE_VERSION,
    "run_mode": RUN_MODE,
    "model_type": (
        "stacked_lstm_language_model"
    ),
    "parameter_count": int(
        MODEL_METADATA[
            "parameter_count"
        ]
    ),
    "parameter_millions": float(
        MODEL_METADATA[
            "parameter_millions"
        ]
    ),
    "training_status": (
        TRAINING_SUMMARY[
            "status"
        ]
    ),
    "optimizer_steps": int(
        TRAINING_SUMMARY[
            "optimizer_steps"
        ]
    ),
    "tokens_seen": int(
        TRAINING_SUMMARY[
            "tokens_seen"
        ]
    ),
    "maximum_training_steps": int(
        TRAINING_SUMMARY[
            "max_steps"
        ]
    ),
    "best_validation_loss": float(
        TRAINING_SUMMARY[
            "best_validation_loss"
        ]
    ),
    "best_validation_perplexity": float(
        TRAINING_SUMMARY[
            "best_validation_perplexity"
        ]
    ),
    "best_validation_step": int(
        TRAINING_SUMMARY[
            "best_validation_step"
        ]
    ),
    "early_stopping": (
        TRAINING_SUMMARY[
            "early_stopping"
        ]
    ),
    "prediction_count": int(
        SUBMISSION_METADATA[
            "prediction_count"
        ]
    ),
    "target_words_per_prediction": int(
        SUBMISSION_METADATA[
            "target_words"
        ]
    ),
    "checkpoint_files": CHECKPOINT_FILES,
    "artifacts": ARTIFACT_INVENTORY,
    "submission_root_file": "prediction.json",
    "official_test_requirements": {
        "all_ids_present": True,
        "exact_word_count_validated": True,
        "prompt_excluded_from_output": True,
    },
    "status": "complete",
}

FINAL_MANIFEST_PATH = (
    METRICS_DIR /
    "architecture_C_final_manifest.json"
)

with open(
    FINAL_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        FINAL_EXPERIMENT_MANIFEST,
        f,
        ensure_ascii=False,
        indent=2,
    )

# -------------------------------------------------------------------------
# Final runtime summary.
# -------------------------------------------------------------------------
print(
    "\n✅ Architecture-C experiment finalized."
)

print(
    f"Architecture       : "
    f"{ARCHITECTURE_NAME}"
)

print(
    f"Run mode           : "
    f"{RUN_MODE}"
)

print(
    f"Parameters         : "
    f"{MODEL_METADATA['parameter_count']:,} "
    f"({MODEL_METADATA['parameter_millions']:.2f}M)"
)

print(
    f"Training status    : "
    f"{TRAINING_SUMMARY['status']}"
)

print(
    f"Steps completed    : "
    f"{TRAINING_SUMMARY['optimizer_steps']:,}"
)

print(
    f"Tokens seen        : "
    f"{TRAINING_SUMMARY['tokens_seen']:,}"
)

print(
    f"Best validation    : "
    f"{TRAINING_SUMMARY['best_validation_loss']:.6f}"
)

print(
    f"Best validation PPL: "
    f"{TRAINING_SUMMARY['best_validation_perplexity']:.4f}"
)

print(
    f"Predictions        : "
    f"{SUBMISSION_METADATA['prediction_count']:,}"
)

print(
    f"Target words       : "
    f"{SUBMISSION_METADATA['target_words']}"
)

print(
    f"Submission ZIP     : "
    f"{SUBMISSION_DIR / 'submission.zip'}"
)

print(
    f"Final manifest     : "
    f"{FINAL_MANIFEST_PATH}"
)

🧾 Finalizing Architecture-C experiment manifest...

✅ Architecture-C experiment finalized.
Architecture       : Architecture_C_Deep_LSTM_Language_Model
Run mode           : production
Parameters         : 103,404,880 (103.40M)
Training status    : max_steps_reached
Steps completed    : 12,208
Tokens seen        : 100,007,936
Best validation    : 3.870509
Best validation PPL: 47.9668
Predictions        : 1,000
Target words       : 128
Submission ZIP     : /kaggle/working/architecture_C/submission/submission.zip
Final manifest     : /kaggle/working/architecture_C/metrics/architecture_C_final_manifest.json
